In [ ]:
# =============================================================================
# jcp-4466772 — CELLULE UNIQUE — espace de features large, cibles du manuscrit
#
# CIBLES (Table 1 du manuscrit) : NPM 4500 mal / 1500 ben, PyPI 998 mal / 1000 ben.
# Le code les vise et RAPPORTE l'ecart. Contrainte externe connue : les packages
# malveillants retires du registre renvoient 404, donc le rendement de collecte
# est d'environ 14% cote NPM et 1,4% cote PyPI. 4500 malveillants NPM est
# atteignable en augmentant les tentatives ; 998 malveillants PyPI ne l'est pas
# (OSV ne contient que ~11 600 avis PyPI). L'ecart figure dans R['collection'].
#
# ESPACE DE FEATURES : introspection automatique du MANIFESTE DE LA VERSION
# (NPM versions[v] ; PyPI /pypi/{pkg}/{version}/json), qui est fige a la
# publication -> l'introspection redevient point-in-time correcte. Liste noire
# explicite des champs post-hoc. L'inventaire complet est exporte (R2.1).
#
# v7 : suite d'ablations complete. Les deux ablations de la v6 etaient muettes
# par construction (variance testee apres le pre-filtre, ordre ecrase par le
# budget). Neuf experiences remplacent les deux : courbe cumulative, leave-one-
# stage-out, variance sur l'espace complet, ordre mesure avant ET apres budget,
# comparaison a budget egal contre cinq selecteurs alternatifs et un tirage
# aleatoire, courbe de budget, stabilite de la selection par Jaccard, ablation
# par famille de metadonnees, sensibilite au pre-filtre ecosysteme.
#
# AUCUNE de ces experiences n'est concue pour produire un resultat favorable.
# Le bilan final indique explicitement si l'avantage du pipeline hybride est
# distinguable du bruit ; s'il ne l'est pas, la contribution doit etre presentee
# comme empirique (compressibilite de l'espace de features) et non
# methodologique.
#
# v6 : budget de 18 features declare a priori (Section 2.3) et retrait des
# features constantes dans un ecosysteme avant la selection du protocole B.
#
# CORRECTIFS v5 : bug de seuil (fpr_upper renvoyait nan quand FP = n_benins, et
# min() sur nan choisissait le seuil le plus bas -> tout classe malveillant),
# collecte des benins NPM via la liste globale (l'API de recherche plafonnait a
# ~550 noms), tirage de version DANS la fenetre temporelle des malveillants,
# RNG par package, grid search reel en CV groupee, latence instrumentee,
# calibration (Brier + ECE + fiabilite), robustesse avec definitions et
# repetitions, controle apparie sur la maturite, part du signal attribuable a
# l'ecosysteme quantifiee.
#
# DEUX PROTOCOLES sur les memes donnees :
#   A : features vman_* + pit_* + cur_*, split aleatoire, selection et scaler
#       sur tout le jeu                    -> reproduit le protocole soumis
#   B : features vman_* + pit_* seules, split group-aware, selection et scaler
#       sur le train uniquement            -> protocole defendable
# L'ecart A - B mesure l'inflation due aux fuites (R1.2, R1.3).
# =============================================================================
!pip install -q pandas numpy requests tqdm scikit-learn catboost shap boruta scipy

import os, re, json, time, random, shutil, warnings, itertools
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt, seaborn as sns
from datetime import datetime, timezone
from collections import Counter
from tqdm.auto import tqdm
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif, RFE
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             matthews_corrcoef, roc_auc_score, average_precision_score,
                             confusion_matrix, brier_score_loss)
from scipy.stats import binomtest, spearmanr, beta as _beta
from catboost import CatBoostClassifier
from boruta import BorutaPy
import shap

warnings.filterwarnings("ignore")
SEED = 42; random.seed(SEED); np.random.seed(SEED)

CFG = dict(
    targets={"npm": {1: 4500, 0: 4000}, "pypi": {1: 998, 0: 1000}},
    attempt_multiplier={"npm": 12, "pypi": 25},   # 404 sur les packages retires
    benign_attempt_multiplier=6,                  # versions hors fenetre temporelle
    name_sim=0.85, test_frac=0.25, val_frac=0.20,
    var_thr=0.01, mi_topk=100, boruta_iter=50, boruta_alpha=0.05,
    rfe_target=30, rfe_step=5, corr_cut=0.85,
    n_boot=1000, prevalences=[0.01, 0.001, 0.0001],
    alert_budget_10k=50, min_recall=0.80,
    final_feature_budget=18,              # budget declare A PRIORI (Section 2.3)
    drop_single_ecosystem_features=True,  # anti-confondant ecosysteme
    budget_sweep=[5, 10, 15, 18, 25, 30, 40],   # courbe performance/compacite
    n_random_baseline=20,                 # tirages aleatoires a budget egal
    n_stability_runs=10,                  # Jaccard des ensembles selectionnes
    min_malicious_for_window=50, str_cap=200, size_cap=50,
)
try:
    from google.colab import drive; drive.mount('/content/drive')
    OUT = "/content/drive/MyDrive/JCP_revision"
except Exception:
    OUT = "./JCP_revision"
os.makedirs(OUT, exist_ok=True)
CACHE = f"{OUT}/dataset_v5.csv"   # collecte inchangee depuis v5 : cache reutilisable
R = {"config": CFG}
print("Sorties :", OUT)

# =============================================================================
# INTROSPECTION DU MANIFESTE — reproduit extract_safe_metadata du manuscrit,
# mais appliquee au manifeste FIGE de la version evaluee.
# =============================================================================
# Champs ajoutes ou modifies APRES la publication : ils encodent le label.
MANIFEST_BLACKLIST = {
    "deprecated",          # ajoute quand le package est signale
    "_npmUser", "_hasShrinkwrap", "_nodeVersion", "_npmVersion", "_npmOperationalInternal",
    "yanked", "yanked_reason",
    "downloads", "last_serial",
}
LEAK_PATTERNS = ["exists", "error_type", "downloads", "last_serial", "yanked",
                 "deprecated", "status", "is_removed", "_npmUser"]

def introspect(d, prefix):
    """Aplatit un dict en features numeriques. Meme logique que le manuscrit."""
    out = {}
    if not isinstance(d, dict): return out
    for k, v in d.items():
        if k in MANIFEST_BLACKLIST: continue
        key = re.sub(r"[^0-9a-zA-Z_]", "_", str(k))
        if isinstance(v, dict):
            out[f"{prefix}{key}_dict_size"] = min(len(v), CFG["size_cap"])
            out[f"{prefix}{key}_nested"] = int(any(isinstance(x, (dict, list)) for x in v.values()))
        elif isinstance(v, list):
            out[f"{prefix}{key}_list_size"] = min(len(v), CFG["size_cap"])
        elif isinstance(v, bool):
            out[f"{prefix}{key}_bool"] = int(v)
        elif isinstance(v, (int, float)):
            out[f"{prefix}{key}_num"] = float(max(min(v, 1e6), -1e6))
        elif isinstance(v, str):
            out[f"{prefix}{key}_len"] = min(len(v), CFG["str_cap"])
            out[f"{prefix}{key}_ent"] = _H(v[:CFG["str_cap"]])
    return out

def _ts(s):
    try: return datetime.fromisoformat(str(s).replace("Z","+00:00")).replace(tzinfo=timezone.utc)
    except Exception: return None

def _H(s):
    if not s: return 0.0
    n = len(s); return float(-sum((c/n)*np.log2(c/n) for c in Counter(s).values()))

def _email_score(e):
    if not e or "@" not in str(e): return 0
    d = str(e).rsplit("@",1)[-1].lower()
    if d in {"mailinator.com","guerrillamail.com","10minutemail.com","tempmail.com","yopmail.com"}: return 3
    if d in {"gmail.com","yahoo.com","hotmail.com","outlook.com","qq.com","163.com","mail.ru"}: return 2
    return 1

def _vfeat(v):
    v = str(v or ""); nums = [int(x) for x in re.findall(r"\d+", v)]
    return {"pit_ver_len": len(v), "pit_ver_n_components": len(v.split(".")),
            "pit_ver_has_prerelease": int(bool(re.search(r"[-+]", v))),
            "pit_ver_is_zero_major": int(v.startswith("0.")),
            "pit_ver_is_initial": int(v in {"0.0.1","0.1.0","1.0.0","0.0.0"}),
            "pit_ver_max_component": max(nums or [0]), "pit_ver_entropy": _H(v)}

def _hist(all_ts, t_p):
    pr = sorted(t for t in all_ts if t and t < t_p)     # strictement anterieures a t_p
    f = {"pit_n_prior_releases": len(pr)}
    if pr:
        f["pit_days_since_first"] = (t_p - pr[0]).total_seconds()/86400
        f["pit_days_since_prev"]  = (t_p - pr[-1]).total_seconds()/86400
        f["pit_releases_30d"]  = sum(1 for t in pr if (t_p - t).days <= 30)
        f["pit_releases_365d"] = sum(1 for t in pr if (t_p - t).days <= 365)
        g = np.diff([t.timestamp() for t in pr])/86400 if len(pr) > 1 else np.array([0.0])
        f["pit_median_gap_days"] = float(np.median(g)); f["pit_std_gap_days"] = float(np.std(g))
    else:
        f.update({"pit_days_since_first":0.0,"pit_days_since_prev":0.0,"pit_releases_30d":0,
                  "pit_releases_365d":0,"pit_median_gap_days":0.0,"pit_std_gap_days":0.0})
    return f

def _name(pkg):
    return {"pit_name_len": len(pkg), "pit_name_entropy": _H(pkg),
            "pit_name_has_scope": int(pkg.startswith("@")),
            "pit_name_n_hyphens": pkg.count("-"),
            "pit_name_n_digits": sum(c.isdigit() for c in pkg),
            "pit_name_n_dots": pkg.count(".")}

def npm_features(pkg, version=None, t_min=None, t_max=None):
    try: d = requests.get(f"https://registry.npmjs.org/{requests.utils.quote(pkg, safe='')}",
                          timeout=25).json()
    except Exception: return None
    if "error" in d or "versions" not in d: return None
    V, T = d.get("versions", {}), d.get("time", {})
    if version is None:
        c = [v for v in V if _ts(T.get(v))]
        if t_min is not None or t_max is not None:
            # appariement temporel PAR CONSTRUCTION : on tire la version parmi
            # celles publiees dans la fenetre des malveillants (R1.4). Evite de
            # perdre 60% des benins a un filtrage post-hoc.
            cw = [v for v in c
                  if (t_min is None or _ts(T.get(v)) >= t_min)
                  and (t_max is None or _ts(T.get(v)) <= t_max)]
            c = cw or c
        if not c: return None
        # RNG par package : random.Random(SEED) reinitialise donnait le MEME
        # indice positionnel a tous les packages de meme longueur de liste.
        version = random.Random(hash(pkg) ^ SEED).choice(c)
    if version not in V: return None
    t_p = _ts(T.get(version))
    if t_p is None: return None
    vd = V[version] or {}
    au = vd.get("author") or {}; au = {"email": au} if isinstance(au, str) else au
    mt = vd.get("maintainers") or []; sc = vd.get("scripts") or {}
    f = {"package": pkg, "version": version, "ecosystem": "npm", "t_p": t_p.isoformat(),
         "maintainer": (mt[0].get("name") or mt[0].get("email") or "")
                       if mt and isinstance(mt[0], dict) else ""}
    f.update(introspect(vd, "vman_"))                 # manifeste fige a la publication
    f.update(_vfeat(version)); f.update(_hist([_ts(T.get(v)) for v in V], t_p)); f.update(_name(pkg))
    f.update({
        "pit_n_dependencies": len(vd.get("dependencies") or {}),
        "pit_n_dev_dependencies": len(vd.get("devDependencies") or {}),
        "pit_n_peer_dependencies": len(vd.get("peerDependencies") or {}),
        "pit_n_scripts": len(sc),
        "pit_has_install_hook": int(any(k in sc for k in ("preinstall","install","postinstall"))),
        "pit_scripts_total_len": sum(len(str(v)) for v in sc.values()),
        "pit_scripts_max_len": max([len(str(v)) for v in sc.values()] or [0]),
        "pit_description_len": len(vd.get("description") or ""),
        "pit_description_entropy": _H(vd.get("description") or ""),
        "pit_n_keywords": len(vd.get("keywords") or []),
        "pit_has_license": int(bool(vd.get("license"))),
        "pit_license_len": len(str(vd.get("license") or "")),
        "pit_has_repository": int(bool(vd.get("repository"))),
        "pit_has_homepage": int(bool(vd.get("homepage"))),
        "pit_has_bugs_url": int(bool(vd.get("bugs"))),
        "pit_n_maintainers": len(mt),
        "pit_author_email_domain_score": _email_score(au.get("email")),
        "pit_has_engines": int(bool(vd.get("engines"))),
        "pit_has_files_field": int(bool(vd.get("files"))),
        "pit_n_classifiers": 0, "pit_requires_python_len": 0, "pit_long_description_len": 0,
        # --- etat ACTUEL du registre : protocole A uniquement ---
        "cur_n_versions_now": len(V), "cur_n_releases_now": len(T),
        "cur_n_maintainers_now": len(d.get("maintainers") or []),
        "cur_readme_len_now": len(str(d.get("readme") or "")),
        "cur_days_since_modified": ((datetime.now(timezone.utc) - _ts(T.get("modified"))).days
                                    if _ts(T.get("modified")) else 0),
        "cur_project_url_len_now": 0, "cur_description_len_now": 0, "cur_homepage_len_now": 0,
    })
    return f

def pypi_features(pkg, version=None, t_min=None, t_max=None):
    try: d = requests.get(f"https://pypi.org/pypi/{requests.utils.quote(pkg, safe='')}/json",
                          timeout=25).json()
    except Exception: return None
    rel = d.get("releases") or {}
    if not rel: return None
    def rts(v):
        fl = rel.get(v) or []
        return _ts(fl[0].get("upload_time_iso_8601")) if fl else None
    if version is None:
        c = [v for v in rel if rts(v)]
        if t_min is not None or t_max is not None:
            cw = [v for v in c
                  if (t_min is None or rts(v) >= t_min)
                  and (t_max is None or rts(v) <= t_max)]
            c = cw or c
        if not c: return None
        version = random.Random(hash(pkg) ^ SEED).choice(c)
    t_p = rts(version)
    if t_p is None: return None
    # metadonnees SPECIFIQUES A LA VERSION : point-in-time correct
    try:
        dv = requests.get(f"https://pypi.org/pypi/{requests.utils.quote(pkg, safe='')}/"
                          f"{requests.utils.quote(str(version), safe='')}/json", timeout=25).json()
        info = dv.get("info") or {}
    except Exception:
        info = d.get("info") or {}
    reqs = info.get("requires_dist") or []
    kw = info.get("keywords") or ""; kw = kw.split(",") if isinstance(kw, str) else list(kw)
    f = {"package": pkg, "version": version, "ecosystem": "pypi", "t_p": t_p.isoformat(),
         "maintainer": info.get("author") or info.get("maintainer") or ""}
    f.update(introspect(info, "vman_"))               # info de CETTE version
    f.update(_vfeat(version)); f.update(_hist([rts(v) for v in rel], t_p)); f.update(_name(pkg))
    f.update({
        "pit_n_dependencies": len(reqs),
        "pit_n_dev_dependencies": sum(1 for r in reqs if "extra ==" in str(r)),
        "pit_n_peer_dependencies": 0,
        "pit_n_scripts": 0, "pit_has_install_hook": 0,
        "pit_scripts_total_len": 0, "pit_scripts_max_len": 0,
        "pit_description_len": len(info.get("summary") or ""),
        "pit_description_entropy": _H(info.get("summary") or ""),
        "pit_long_description_len": len(info.get("description") or ""),
        "pit_n_keywords": len([k for k in kw if str(k).strip()]),
        "pit_has_license": int(bool(info.get("license"))),
        "pit_license_len": len(str(info.get("license") or "")),
        "pit_has_repository": int(bool(info.get("project_urls") or {})),
        "pit_has_homepage": int(bool(info.get("home_page"))),
        "pit_has_bugs_url": int("tracker" in json.dumps(info.get("project_urls") or {}).lower()),
        "pit_n_maintainers": int(bool(info.get("maintainer") or info.get("author"))),
        "pit_author_email_domain_score": _email_score(info.get("author_email")),
        "pit_has_engines": 0, "pit_has_files_field": 0,
        "pit_n_classifiers": len(info.get("classifiers") or []),
        "pit_requires_python_len": len(str(info.get("requires_python") or "")),
        # --- etat ACTUEL du registre : protocole A uniquement ---
        "cur_n_versions_now": len(rel),
        "cur_n_releases_now": sum(len(v or []) for v in rel.values()),
        "cur_n_maintainers_now": int(bool((d.get("info") or {}).get("maintainer"))),
        "cur_readme_len_now": len((d.get("info") or {}).get("description") or ""),
        "cur_days_since_modified": 0,
        "cur_project_url_len_now": len(str((d.get("info") or {}).get("project_url") or "")),
        "cur_description_len_now": len((d.get("info") or {}).get("description") or ""),
        "cur_homepage_len_now": len(str((d.get("info") or {}).get("home_page") or "")),
    })
    return f

# =============================================================================
# COLLECTE
# =============================================================================
BEHAVIOURS = ["exfiltrat","credential","token","steal","backdoor","reverse shell",
              "install hook","preinstall","postinstall","wallet","clipboard",
              "typosquat","dependency confusion","remote code","payload","miner"]
NPM_KW = ["utils","test","lib","api","cli","http","auth","parser","logger","config",
          "stream","crypto","date","format","validate","router","cache","queue","socket",
          "template","math","string","array","object","file","path","url","json","yaml",
          "csv","image","color","random","uuid","hash","compress","server","client",
          "proxy","middleware","plugin","build","bundle","lint","mock","fixture","assert",
          "benchmark","docs","types"]

def collect_malicious_versions(eco, limit):
    rows, seen, excl = [], set(), Counter()
    files = []
    for root, _, fs in os.walk(f"malicious-packages/osv/malicious/{eco}"):
        files += [os.path.join(root, f) for f in fs if f.endswith(".json")]
    random.shuffle(files)
    for fp in tqdm(files, desc=f"OSV {eco}"):
        if len(rows) >= limit: break
        try: d = json.load(open(fp, encoding="utf-8"))
        except Exception: excl["json_unreadable"] += 1; continue
        if not any(b in json.dumps(d).lower() for b in BEHAVIOURS):
            excl["no_attributed_behaviour"] += 1; continue
        for a in d.get("affected", []):
            pk = a.get("package", {})
            if pk.get("ecosystem","").lower() != eco or not pk.get("name"): continue
            vs = a.get("versions") or []
            if not vs: excl["no_explicit_version"] += 1; continue
            for v in vs:
                if (pk["name"], v) in seen: excl["duplicate"] += 1; continue
                seen.add((pk["name"], v)); rows.append({"package": pk["name"], "version": v})
                if len(rows) >= limit: break
    return pd.DataFrame(rows), dict(excl)

def collect_npm_benign_bulk(target):
    """L'API de recherche NPM ne pagine pas au-dela d'environ 250 resultats par
    requete : 50 mots-cles n'avaient produit que 548 noms uniques pour une cible
    de 1500. On passe par la liste publique de tous les noms de packages NPM."""
    names = set()
    try:
        r = requests.get("https://raw.githubusercontent.com/nice-registry/"
                         "all-the-package-names/master/names.json", timeout=300)
        if r.status_code == 200:
            names.update(json.loads(r.text))
    except Exception as e:
        print("  liste globale NPM indisponible :", e)
    if not names:
        print("  repli sur l'API de recherche par mot-cle")
        return collect_npm_benign_search(target)
    names = sorted(names); random.Random(SEED).shuffle(names)
    print(f"  {len(names)} noms NPM disponibles")
    return names[: target * CFG["benign_attempt_multiplier"]]

def collect_npm_benign_search(limit):
    """Repli : recherche par mot-cle (plafonnee)."""
    s = set()
    for t in tqdm(NPM_KW, desc="NPM search"):
        if len(s) >= limit * 4: break
        for start in range(0, 1000, 50):
            try:
                r = requests.get("https://registry.npmjs.org/-/v1/search",
                                 params={"text": t, "size": 50, "from": start}, timeout=20).json()
            except Exception: break
            o = r.get("objects", [])
            if not o: break
            s.update(x["package"]["name"] for x in o)
            if len(s) >= limit * 4: break
            time.sleep(0.15)
    return sorted(s)

def collect_pypi_benign(limit):
    s, excl = set(), Counter()
    try:
        r = requests.get("https://hugovk.github.io/top-pypi-packages/top-pypi-packages-30-days.json",
                         timeout=30).json()
        s.update(p["project"] for p in r.get("rows", [])[:1000])
    except Exception: excl["top_list_unreachable"] += 1
    try:
        allp = re.findall(r">([^<]+)</a>", requests.get("https://pypi.org/simple/", timeout=180).text)
        random.shuffle(allp); s.update(p.strip() for p in allp[:limit*4])
    except Exception: excl["simple_index_unreachable"] += 1

    return sorted(s), dict(excl)

def build(items, eco, label, target, desc, t_min=None, t_max=None):
    fn = npm_features if eco == "npm" else pypi_features
    out, seen, tried, failed = [], set(), 0, 0
    it = items if isinstance(items, list) else items.to_dict("records")
    bar = tqdm(it, desc=desc)
    for x in bar:
        if len(out) >= target: break
        pkg, ver = (x, None) if isinstance(x, str) else (x["package"], x.get("version"))
        if (pkg, ver) in seen: continue
        seen.add((pkg, ver)); tried += 1
        r_ = fn(pkg, ver, t_min, t_max)
        if r_: r_["label"] = label; out.append(r_)
        else:  failed += 1
        if tried % 200 == 0: bar.set_postfix(obtenus=len(out), cible=target)
    return pd.DataFrame(out), {"attempted": tried, "obtained": len(out),
                               "unretrievable_404_or_error": failed, "target": target,
                               "shortfall": max(0, target - len(out))}

if os.path.exists(CACHE):
    df = pd.read_csv(CACHE, low_memory=False)
    print(f"Jeu charge depuis le cache : {CACHE}")
    R["collection"] = {"loaded_from_cache": True}
else:
    for p in ["malicious-packages","malicious-packages-main","ossf.zip"]:
        if os.path.exists(p): shutil.rmtree(p) if os.path.isdir(p) else os.remove(p)
    !wget -q https://github.com/ossf/malicious-packages/archive/refs/heads/main.zip -O ossf.zip
    !unzip -q ossf.zip
    os.rename("malicious-packages-main", "malicious-packages")

    TG, AM = CFG["targets"], CFG["attempt_multiplier"]

    # --- 1) MALVEILLANTS d'abord : ils definissent la fenetre temporelle -----
    npm_mal_df,  npm_excl  = collect_malicious_versions("npm",  TG["npm"][1]  * AM["npm"])
    pypi_mal_df, pypi_excl = collect_malicious_versions("pypi", TG["pypi"][1] * AM["pypi"])
    d1, s1 = build(npm_mal_df,  "npm",  1, TG["npm"][1],  "npm-mal")
    d2, s2 = build(pypi_mal_df, "pypi", 1, TG["pypi"][1], "pypi-mal")

    WIN = {}
    for dd, eco in [(d1, "npm"), (d2, "pypi")]:
        if len(dd) >= CFG["min_malicious_for_window"]:
            t = pd.to_datetime(dd["t_p"], utc=True, errors="coerce").dropna()
            WIN[eco] = (t.quantile(.05), t.quantile(.95))
        else:
            WIN[eco] = (None, None)
    print("\nFenetres temporelles issues des malveillants :",
          {k: [str(v[0].date()), str(v[1].date())] if v[0] is not None else "aucune"
           for k, v in WIN.items()})

    # --- 2) BENINS, tires DANS la fenetre de leur ecosysteme (R1.4) ----------
    print("\nCollecte de benins NPM (liste globale) :")
    npm_ben_names = collect_npm_benign_bulk(TG["npm"][0])
    pypi_ben_names, pypi_ben_excl = collect_pypi_benign(TG["pypi"][0])

    mal_names = set(npm_mal_df.package) | set(pypi_mal_df.package)
    n0 = (len(npm_ben_names), len(pypi_ben_names))
    npm_ben_names  = [p for p in npm_ben_names  if p not in mal_names]
    pypi_ben_names = [p for p in pypi_ben_names if p not in mal_names]
    assert len(npm_ben_names)  == len(set(npm_ben_names))
    assert len(pypi_ben_names) == len(set(pypi_ben_names))

    d3, s3 = build(npm_ben_names,  "npm",  0, TG["npm"][0],  "npm-ben",
                   t_min=WIN["npm"][0],  t_max=WIN["npm"][1])
    d4, s4 = build(pypi_ben_names, "pypi", 0, TG["pypi"][0], "pypi-ben",
                   t_min=WIN["pypi"][0], t_max=WIN["pypi"][1])

    df = pd.concat([d1, d2, d3, d4], ignore_index=True)
    df = df.drop_duplicates(subset=["package","version","ecosystem"]).reset_index(drop=True)
    df.to_csv(CACHE, index=False)
    R["collection"] = {
        "yield": {"npm_malicious": s1, "pypi_malicious": s2,
                  "npm_benign": s3, "pypi_benign": s4},
        "osv_exclusions": {"npm": npm_excl, "pypi": pypi_excl},
        "pypi_benign_exclusions": pypi_ben_excl,
        "npm_benign_candidates_before_filter": n0[0],
        "pypi_benign_candidates_before_filter": n0[1],
        "temporal_windows_from_malicious": {
            k: [str(v[0]), str(v[1])] if v[0] is not None else None for k, v in WIN.items()},
        "npm_keywords": NPM_KW, "behaviour_terms": BEHAVIOURS, "loaded_from_cache": False}
    print("\nRENDEMENT DE COLLECTE (cible -> obtenu) :")
    for k, s in R["collection"]["yield"].items():
        print(f"  {k:15s} cible {s['target']:5d} | obtenu {s['obtained']:5d} | "
              f"tentatives {s['attempted']:6d} | manquant {s['shortfall']:5d}")
    print("  Les manquants viennent des packages retires du registre (404), non d'un choix.")

df["t_p"] = pd.to_datetime(df["t_p"], utc=True, errors="coerce")
df = df.dropna(subset=["t_p"]).reset_index(drop=True)
df = df.fillna(0)
print("\nDimensions :", df.shape)
print(df.groupby(["ecosystem","label"]).size())

# Inventaire des features -> Supplementary Table S3 (R1.6, R2.1)
META = ["package","version","ecosystem","t_p","maintainer","label","group_id","canon"]
allfeat = [c for c in df.columns if c not in META]
inv = pd.DataFrame({"feature": allfeat,
                    "family": ["vman (manifeste de la version)" if c.startswith("vman_")
                               else "pit (reconstruite a t_p)" if c.startswith("pit_")
                               else "cur (etat actuel du registre)" if c.startswith("cur_")
                               else "autre" for c in allfeat],
                    "point_in_time": [not c.startswith("cur_") for c in allfeat],
                    "n_unique": [int(df[c].nunique()) if c in df.columns else 0 for c in allfeat]})
inv.to_csv(f"{OUT}/supplementary_S3_feature_inventory.csv", index=False)
print(f"\nEspace de features : {len(allfeat)} "
      f"(vman {sum(c.startswith('vman_') for c in allfeat)}, "
      f"pit {sum(c.startswith('pit_') for c in allfeat)}, "
      f"cur {sum(c.startswith('cur_') for c in allfeat)})")
R["feature_space"] = {"total": len(allfeat),
                      "vman": int(sum(c.startswith("vman_") for c in allfeat)),
                      "pit": int(sum(c.startswith("pit_") for c in allfeat)),
                      "cur": int(sum(c.startswith("cur_") for c in allfeat))}

# =============================================================================
# FENETRE TEMPORELLE PAR ECOSYSTEME + GROUPES
# =============================================================================
parts, win = [], {}
for eco in df.ecosystem.unique():
    sub = df[df.ecosystem == eco]; mal = sub[sub.label == 1]
    if len(mal) < CFG["min_malicious_for_window"]:
        parts.append(sub); win[eco] = "aucune (trop peu de malveillants)"; continue
    lo_e, hi_e = mal.t_p.quantile(.05), mal.t_p.quantile(.95)
    parts.append(sub[(sub.t_p >= lo_e) & (sub.t_p <= hi_e)])
    win[eco] = [str(lo_e.date()), str(hi_e.date())]
nb0 = len(df); df = pd.concat(parts, ignore_index=True)
print(f"\nAppariement temporel : {nb0} -> {len(df)} versions | {win}")

canon = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
def nsim(a, b):
    if a == b: return 1.0
    la, lb = len(a), len(b)
    if not la or not lb: return 0.0
    prev = list(range(lb+1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j]+1, cur[j-1]+1, prev[j-1]+(ca != cb)))
        prev = cur
    return 1.0 - prev[lb]/max(la, lb)
class UF:
    def __init__(s): s.p = {}
    def find(s, x):
        s.p.setdefault(x, x)
        while s.p[x] != x: s.p[x] = s.p[s.p[x]]; x = s.p[x]
        return x
    def union(s, a, b):
        ra, rb = s.find(a), s.find(b)
        if ra != rb: s.p[ra] = rb
uf = UF(); df["canon"] = df["package"].map(canon)
for _, r_ in df.iterrows():
    uf.find(("pkg", r_["canon"]))
    m = str(r_.get("maintainer") or "").strip().lower()
    if m and m not in ("nan", "0"): uf.union(("pkg", r_["canon"]), ("mnt", m))
blocks = {}
for n in sorted(df["canon"].unique()): blocks.setdefault((n[:2], len(n)//4), []).append(n)
npairs = 0
for blk in tqdm(blocks.values(), desc="familles de noms"):
    for a, b in itertools.combinations(blk, 2):
        if abs(len(a)-len(b)) <= 3 and nsim(a, b) >= CFG["name_sim"]:
            uf.union(("pkg", a), ("pkg", b)); npairs += 1
df["group_id"] = df["canon"].map(lambda n: str(uf.find(("pkg", n))))
sz = df.group_id.value_counts()
print(f"{len(df)} versions | {df.package.nunique()} packages | {df.group_id.nunique()} groupes "
      f"| plus grand : {sz.iloc[0]} ({sz.iloc[0]/len(df):.1%})")
if sz.iloc[0] > 0.10*len(df):
    print("!! Un groupe depasse 10% du jeu : le split B sera desequilibre.")
R["dataset"] = {"n_versions": int(len(df)), "n_packages": int(df.package.nunique()),
                "n_groups": int(df.group_id.nunique()), "largest_group": int(sz.iloc[0]),
                "similar_name_pairs": int(npairs), "windows": win,
                "composition": {f"{e}_{l}": int(n)
                                for (e,l), n in df.groupby(["ecosystem","label"]).size().items()}}
print(R["dataset"])

# =============================================================================
# FONCTIONS COMMUNES
# =============================================================================
def eco_proxy_features(part, prefixes):
    """Features constantes dans au moins un ecosysteme : ce sont des indicateurs
    d'ecosysteme deguises. Or l'ecosysteme est correle au label (NPM ~90%
    malveillant, PyPI ~11%), donc ces features encodent partiellement le label
    sans porter aucune information sur la malveillance. Au run precedent, 12 des
    24 features retenues etaient dans ce cas, dont la premiere au classement SHAP."""
    if not CFG["drop_single_ecosystem_features"] or part.ecosystem.nunique() < 2:
        return []
    X = part.drop(columns=[c for c in META if c in part.columns], errors="ignore")
    X = X.select_dtypes(include=[np.number])
    cols = [c for c in X.columns if c.startswith(prefixes)]
    bad = []
    for c in cols:
        st = part.groupby("ecosystem")[c].std().fillna(0)
        if (st == 0).any(): bad.append(c)
    return bad

ECO_PROXIES = []          # rempli avant le protocole B

def fframe(part, prefixes):
    X = part.drop(columns=[c for c in META if c in part.columns], errors="ignore")
    X = X.select_dtypes(include=[np.number])
    keep = [c for c in X.columns if c.startswith(prefixes)
            and not any(re.search(p, c) for p in LEAK_PATTERNS)
            and c not in ECO_PROXIES]
    return X[keep].copy()

def make_prep(train_part, prefixes):
    X0 = fframe(train_part, prefixes); F0 = list(X0.columns)
    imp = SimpleImputer(strategy="median").fit(X0[F0])
    A = imp.transform(X0[F0])
    LOq, HIq = np.nanpercentile(A, 1, 0), np.nanpercentile(A, 99, 0)
    scl = StandardScaler().fit(np.clip(A, LOq, HIq))
    def prep(part):
        X = fframe(part, prefixes).reindex(columns=F0, fill_value=0)
        return pd.DataFrame(scl.transform(np.clip(imp.transform(X), LOq, HIq)),
                            columns=F0, index=part.index)
    return prep, F0

def select(X, y, stages=("variance","mi","boruta","rfe","corr","budget"), verbose=True):
    log, cur = {"initial": X.shape[1]}, list(X.columns)
    def say(k, v):
        log[k] = len(v)
        if verbose: print(f"   {k:>12} -> {len(v)}")
    if verbose: print(f"   {'initial':>12} -> {len(cur)}")
    for st in stages:
        if st == "variance":
            s = VarianceThreshold(CFG["var_thr"]).fit(X[cur])
            cur = [c for c, k in zip(cur, s.get_support()) if k]; say("variance", cur)
        elif st == "mi":
            mi = mutual_info_classif(X[cur], y, random_state=SEED)
            cur = [cur[i] for i in sorted(np.argsort(mi)[::-1][:CFG["mi_topk"]])]; say("mutual_info", cur)
        elif st == "boruta":
            b = BorutaPy(RandomForestClassifier(n_jobs=-1, max_depth=5, random_state=SEED,
                         class_weight="balanced"), n_estimators="auto", verbose=0,
                         random_state=SEED, max_iter=CFG["boruta_iter"], alpha=CFG["boruta_alpha"])
            b.fit(X[cur].values, y)
            k = [c for c, s_ in zip(cur, b.support_) if s_]; cur = k or cur; say("boruta", cur)
        elif st == "rfe":
            k = min(CFG["rfe_target"], len(cur))
            r_ = RFE(CatBoostClassifier(iterations=100, learning_rate=0.1, depth=4, verbose=0,
                     random_state=SEED), n_features_to_select=k, step=CFG["rfe_step"]).fit(X[cur], y)
            cur = [c for c, s_ in zip(cur, r_.support_) if s_]; say("rfe", cur)
        elif st == "corr":
            mi = dict(zip(cur, mutual_info_classif(X[cur], y, random_state=SEED)))
            C = X[cur].corr().abs(); keep, drop = [], set()
            for f_ in sorted(cur, key=lambda z: -mi[z]):
                if f_ in drop: continue
                keep.append(f_)
                for o in C.index[C[f_] > CFG["corr_cut"]]:
                    if o != f_ and o not in keep: drop.add(o)
            cur = keep; say("corr_prune", cur)
        elif st == "budget":
            # Budget declare A PRIORI : c'est une decision de conception, a
            # enoncer en Section 2.3, et non un nombre qui tombe du pipeline.
            # Sans elle, les executions donnaient 17, 23, 24 ou 28 features
            # selon la collecte, alors que le manuscrit annonce 18.
            B = CFG.get("final_feature_budget")
            if B and len(cur) > B:
                m_ = CatBoostClassifier(iterations=300, learning_rate=0.1, depth=6,
                                        verbose=0, random_state=SEED,
                                        auto_class_weights="Balanced").fit(X[cur], y)
                imp_ = pd.Series(m_.get_feature_importance(), index=cur
                                 ).sort_values(ascending=False)
                cur = list(imp_.head(B).index); say("budget", cur)
    return [str(c) for c in cur], log

def metrics(y, s, thr=0.5):
    yp = (s >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yp, labels=[0,1]).ravel()
    assert tn+fp+fn+tp == len(y), "la matrice ne somme pas a la taille du test"
    p  = precision_score(y, yp, pos_label=1, zero_division=0)
    r_ = recall_score(y, yp, pos_label=1, zero_division=0)
    f1 = f1_score(y, yp, pos_label=1, zero_division=0)
    if p + r_ > 0: assert abs(2*p*r_/(p+r_) - f1) < 1e-9, "F1 incoherent avec precision/recall"
    return dict(n=int(len(y)), TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp),
                test_accuracy=accuracy_score(y, yp), test_precision_pos=p, test_recall_pos=r_,
                test_f1_pos=f1, test_mcc=matthews_corrcoef(y, yp),
                test_auc_roc=roc_auc_score(y, s), test_au_pr=average_precision_score(y, s),
                au_pr_baseline=float(np.mean(y)),
                test_specificity=tn/(tn+fp) if tn+fp else np.nan,
                test_fpr=fp/(tn+fp) if tn+fp else np.nan,
                test_brier=brier_score_loss(y, s), threshold=float(thr))

def fpr_upper(fp, n_ben, conf=0.95):
    """Borne superieure du FPR, definie pour TOUS les cas.
    fp = 0      -> regle de trois : un FPR mesure a 0 sur n benins n'est pas 0.
    fp = n_ben  -> 1.0. Ce cas renvoyait nan, et min() sur une liste contenant
                   nan retournait le premier element, donc le seuil le plus bas :
                   le modele classait tout comme malveillant (FPR = 1.0)."""
    if n_ben <= 0:  return np.nan
    if fp <= 0:     return 3.0 / n_ben
    if fp >= n_ben: return 1.0
    return float(_beta.ppf(conf, fp + 1, n_ben - fp))

def prev_table(y, s, thr, pv=CFG["prevalences"]):
    tn, fp, fn, tp = confusion_matrix(y, (s >= thr).astype(int), labels=[0,1]).ravel()
    tpr  = tp/(tp+fn) if tp+fn else np.nan
    fpr  = fp/(fp+tn) if fp+tn else np.nan
    fprU = fpr_upper(fp, fp+tn)
    rows = []
    for pi in pv:
        for lab, f_ in [("point", fpr), ("upper95", fprU)]:
            den = tpr*pi + f_*(1-pi)
            rows.append({"prevalence": pi, "fpr_estimate": lab, "TPR": tpr, "FPR": f_,
                         "PPV": (tpr*pi)/den if den and den > 0 else np.nan,
                         "alerts_per_10k": 10000*den if den == den else np.nan,
                         "true_positives_per_10k": 10000*tpr*pi,
                         "missed_per_10k": 10000*pi*(1-tpr)})
    return pd.DataFrame(rows)

def choose_threshold(y_val, s_val, min_recall=CFG["min_recall"], prevalence=0.001,
                     budget=CFG["alert_budget_10k"]):
    """Deux contraintes : rappel minimal ET budget. Un seuil a rappel 0.13
    respecte le budget mais manque 87% des malveillants."""
    cands = []
    for t in np.unique(np.round(s_val, 3)):
        yp = (s_val >= t).astype(int)
        rec = recall_score(y_val, yp, pos_label=1, zero_division=0)
        if rec < min_recall: continue
        tn, fp, fn, tp = confusion_matrix(y_val, yp, labels=[0,1]).ravel()
        f_ = fpr_upper(fp, fp+tn)
        alerts = 10000*(rec*prevalence + f_*(1-prevalence))
        if not np.isfinite(alerts): continue     # garde-fou : nan faussait le min()
        cands.append((float(t), float(rec), float(alerts), int(fp), int(fp+tn)))
    if not cands:
        return 0.5, False, f"aucun seuil fini n'atteint un rappel >= {min_recall}"
    ok = [c for c in cands if c[2] <= budget]
    if ok:
        b = min(ok, key=lambda z: z[2])
        return b[0], True, f"budget respecte (rappel {b[1]:.3f}, {b[3]}/{b[4]} FP)"
    b = min(cands, key=lambda z: z[2])
    return b[0], False, (f"budget de {budget} alertes/10k inatteignable a rappel >= "
                         f"{min_recall} : minimum realisable {b[2]:.0f} alertes/10k "
                         f"(rappel {b[1]:.3f}, {b[3]}/{b[4]} FP)")

def boot_ci(y, s, fn, n=CFG["n_boot"]):
    rng = np.random.default_rng(SEED); y, s = np.asarray(y), np.asarray(s); v = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) > 1: v.append(fn(y[i], s[i]))
    return [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))]

DEFAULT_HP = {"CatBoost": {"iterations": 500, "learning_rate": 0.1, "depth": 6,
                           "l2_leaf_reg": 3},
              "RandomForest": {"n_estimators": 200, "max_depth": 20,
                               "min_samples_split": 5, "max_features": "sqrt"}}

def build_models(hp=None):
    """hp = valeurs issues du grid search (protocole B). None = configuration par
    defaut, utilisee par le protocole A qui reproduit la soumission."""
    hp = hp or {}
    cb = dict(DEFAULT_HP["CatBoost"]);     cb.update(hp.get("CatBoost", {}))
    rf = dict(DEFAULT_HP["RandomForest"]); rf.update(hp.get("RandomForest", {}))
    m = {"CatBoost": CatBoostClassifier(verbose=0, random_state=SEED,
                                        auto_class_weights="Balanced", **cb),
         "RandomForest": RandomForestClassifier(class_weight="balanced",
                                        random_state=SEED, n_jobs=-1, **rf)}
    m["Stacking"] = StackingClassifier(
        estimators=[("cat", m["CatBoost"]), ("rf", m["RandomForest"])],
        final_estimator=LogisticRegression(C=1.0, penalty="l2", max_iter=1000), cv=5, n_jobs=-1)
    return m

# =============================================================================
# PROTOCOLE A — reproduction du protocole soumis
# =============================================================================
print("\n" + "="*70 + "\nPROTOCOLE A\n" + "="*70)
ECO_PROXIES = []   # le protocole A reproduit la soumission : aucun filtrage
prepA, F0A = make_prep(df, ("vman_","pit_","cur_"))      # scaler sur TOUT : fuite assumee
XA = prepA(df); yA = df.label.values
SEL_A, LOG_A = select(XA, yA)                            # selection sur TOUT : fuite assumee
iA_tr, iA_te = train_test_split(np.arange(len(df)), test_size=CFG["test_frac"],
                                stratify=yA, random_state=SEED)
rowsA, fitA, scoA = [], {}, {}
for nm, m in build_models().items():
    m.fit(XA.iloc[iA_tr][SEL_A], yA[iA_tr]); fitA[nm] = m
    scoA[nm] = m.predict_proba(XA.iloc[iA_te][SEL_A])[:,1]
    row = {"protocol":"A_original","model":nm,
           "train_accuracy": accuracy_score(yA[iA_tr], m.predict(XA.iloc[iA_tr][SEL_A]))}
    row.update(metrics(yA[iA_te], scoA[nm])); row["overfit_gap"] = row["train_accuracy"]-row["test_accuracy"]
    rowsA.append(row)
TA = pd.DataFrame(rowsA)
print(TA[["model","train_accuracy","test_accuracy","test_f1_pos","test_mcc","test_auc_roc",
          "test_au_pr","au_pr_baseline","test_fpr","TN","FP","FN","TP"]].to_string(index=False))
BEST_A = TA.sort_values("test_au_pr", ascending=False).iloc[0]["model"]
thrA, feasA, msgA = choose_threshold(yA[iA_te], scoA[BEST_A])
PTA = prev_table(yA[iA_te], scoA[BEST_A], thrA)
R["protocol_A"] = {"selection_log": LOG_A, "selected": SEL_A, "n_selected": len(SEL_A),
                   "table": TA.to_dict("records"), "best_model": BEST_A,
                   "threshold": thrA, "budget_feasible": bool(feasA), "message": msgA,
                   "prevalence_table": PTA.to_dict("records")}

# =============================================================================
# PROTOCOLE B — group-aware, point-in-time, selection sur le train
# =============================================================================
print("\n" + "="*70 + "\nPROTOCOLE B\n" + "="*70)
i_tv, i_te = next(GroupShuffleSplit(1, test_size=CFG["test_frac"], random_state=SEED)
                  .split(df, df.label, groups=df.group_id))
trval, test = df.iloc[i_tv].copy(), df.iloc[i_te].copy()
i_tr, i_va = next(GroupShuffleSplit(1, test_size=CFG["val_frac"], random_state=SEED)
                  .split(trval, trval.label, groups=trval.group_id))
train, val = trval.iloc[i_tr].copy(), trval.iloc[i_va].copy()
for a, b, nm in [(train,val,"tr/va"), (train,test,"tr/te"), (val,test,"va/te")]:
    assert not (set(a.group_id) & set(b.group_id)), f"fuite de groupe {nm}"
# Proxies d'ecosysteme retires AVANT la selection (R1.9, confondant ecosysteme).
ECO_PROXIES = eco_proxy_features(train, ("vman_","pit_"))
print(f"Proxies d'ecosysteme retires : {len(ECO_PROXIES)}")
if ECO_PROXIES[:12]: print("   exemples :", ECO_PROXIES[:12])
R["ecosystem_proxies_dropped"] = ECO_PROXIES

prepB, F0B = make_prep(train, ("vman_","pit_"))          # scaler sur le TRAIN seul
Xtr, Xva, Xte = prepB(train), prepB(val), prepB(test)
y_tr, y_va, y_te = train.label.values, val.label.values, test.label.values
print("Selection (train uniquement) :")
SEL_B, LOG_B = select(Xtr, y_tr)                         # selection sur le TRAIN seul
Xtr_s, Xva_s, Xte_s = Xtr[SEL_B], Xva[SEL_B], Xte[SEL_B]

# --- Recherche d'hyperparametres REELLE, en CV GROUPEE (R2.1) ----------------
# Les valeurs publiees doivent venir d'une recherche, pas d'une assertion. Une CV
# ordinaire reintroduirait la fuite corrigee par le split group-aware.
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
cv_grp = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
g_train = train.group_id.values
GRIDS = {
 "CatBoost": (CatBoostClassifier(verbose=0, random_state=SEED,
                                 auto_class_weights="Balanced"),
              {"depth": [4, 6, 8], "learning_rate": [0.01, 0.05, 0.1],
               "l2_leaf_reg": [1, 3, 5], "iterations": [500]}),
 "RandomForest": (RandomForestClassifier(class_weight="balanced",
                                         random_state=SEED, n_jobs=-1),
                  {"n_estimators": [100, 200, 300], "max_depth": [10, 20, None],
                   "min_samples_split": [2, 5, 10], "max_features": ["sqrt", "log2"]}),
}
HP = {}; R["hyperparameters"] = {}
print("Grid search (CV groupee) :")
for nm_, (est_, grid_) in GRIDS.items():
    gs = GridSearchCV(est_, grid_, scoring="average_precision", cv=cv_grp, n_jobs=-1)
    gs.fit(Xtr_s, y_tr, groups=g_train)
    HP[nm_] = gs.best_params_
    print(f"   {nm_:14s} -> {gs.best_params_}  (AU-PR CV = {gs.best_score_:.4f})")
    R["hyperparameters"][nm_] = {"grid": grid_, "selected": gs.best_params_,
                                 "cv_score_au_pr": float(gs.best_score_),
                                 "cv": "StratifiedGroupKFold(5)"}
R["hyperparameters"]["Stacking_meta"] = {"grid": {"C": [1.0]},
                                         "selected": {"C": 1.0, "penalty": "l2"}}
R["hyperparameters"]["note"] = ("protocole A : configuration par defaut reproduisant la "
                                "soumission ; protocole B : valeurs ci-dessus")

rowsB, fitB, scoB = [], {}, {}
for nm, m in build_models(HP).items():
    m.fit(Xtr_s, y_tr); fitB[nm] = m
    scoB[nm] = {"test": m.predict_proba(Xte_s)[:,1], "val": m.predict_proba(Xva_s)[:,1]}
    row = {"protocol":"B_corrected","model":nm,
           "train_accuracy": accuracy_score(y_tr, m.predict(Xtr_s))}
    row.update(metrics(y_te, scoB[nm]["test"])); row["overfit_gap"] = row["train_accuracy"]-row["test_accuracy"]
    rowsB.append(row)
TB = pd.DataFrame(rowsB)
dup = TB.duplicated(subset=["test_accuracy","test_f1_pos","test_auc_roc"], keep=False)
if dup.any(): print("!! Metriques identiques :", TB.loc[dup,"model"].tolist())
print(TB[["model","train_accuracy","test_accuracy","test_precision_pos","test_recall_pos",
          "test_f1_pos","test_mcc","test_auc_roc","test_au_pr","au_pr_baseline",
          "test_specificity","test_fpr","TN","FP","FN","TP"]].to_string(index=False))
BEST = TB.sort_values("test_au_pr", ascending=False).iloc[0]["model"]; mbest = fitB[BEST]
ci = {k: {"auc_roc_ci95": boot_ci(y_te, scoB[k]["test"], roc_auc_score),
          "au_pr_ci95":   boot_ci(y_te, scoB[k]["test"], average_precision_score)} for k in fitB}
print("IC 95% :", json.dumps(ci, indent=2))
mc = {}
for a, b in itertools.combinations(fitB, 2):
    pa = (scoB[a]["test"] >= .5).astype(int); pb_ = (scoB[b]["test"] >= .5).astype(int)
    B_ = int(np.sum((pa==y_te) & (pb_!=y_te))); C_ = int(np.sum((pa!=y_te) & (pb_==y_te)))
    mc[f"{a}_vs_{b}"] = {"b":B_, "c":C_,
                         "p_exact": float(binomtest(min(B_,C_), B_+C_, .5).pvalue) if B_+C_ else 1.0}
for k in mc: mc[k]["p_bonferroni"] = min(1.0, mc[k]["p_exact"]*max(len(mc),1))
print("McNemar :", json.dumps(mc, indent=2))
thr, feasible, msg = choose_threshold(y_va, scoB[BEST]["val"])
PT = prev_table(y_te, scoB[BEST]["test"], thr)
mthr = metrics(y_te, scoB[BEST]["test"], thr)
print(f"\n{BEST} | seuil = {thr:.3f} | {msg}")
print(f"  rappel = {mthr['test_recall_pos']:.3f} | FP = {mthr['FP']} sur {mthr['FP']+mthr['TN']} benins")
print(PT.to_string(index=False))
print("-> publier la ligne 'upper95' : un FPR mesure a 0 n'est pas un FPR de 0.")
R["protocol_B"] = {"selection_log": LOG_B, "selected": SEL_B, "n_selected": len(SEL_B),
                   "table": TB.to_dict("records"), "best_model": BEST, "bootstrap_ci": ci,
                   "mcnemar": {"n_comparisons": len(mc), "results": mc},
                   "threshold": thr, "threshold_selected_on": "validation",
                   "budget_feasible": bool(feasible), "message": msg,
                   "metrics_at_threshold": mthr, "prevalence_table": PT.to_dict("records"),
                   "split": {"n_train": len(train), "n_val": len(val), "n_test": len(test),
                             "test_malicious": int(test.label.sum()),
                             "test_benign": int((1-test.label).sum())}}

# =============================================================================
# A vs B, CONFONDANT ECOSYSTEME, GENERALISATION, ABLATIONS, EVASION, SHAP
# =============================================================================
CMP = pd.concat([TA, TB], ignore_index=True)[
    ["protocol","model","test_accuracy","test_f1_pos","test_mcc","test_auc_roc",
     "test_au_pr","au_pr_baseline","test_fpr","overfit_gap"]]
print("\n" + "="*70 + "\nA vs B\n" + "="*70)
print(CMP.to_string(index=False))
delta = (TA.set_index("model")["test_accuracy"] - TB.set_index("model")["test_accuracy"])
print("\nInflation d'accuracy due aux fuites (A - B) :\n", delta.round(4).to_string())
R["comparison_A_vs_B"] = {"table": CMP.to_dict("records"),
                          "accuracy_inflation": delta.round(6).to_dict()}

comp = df.groupby(["ecosystem","label"]).size().unstack(fill_value=0)
auc_eco = roc_auc_score(df.label.values, df.ecosystem.map(df.groupby("ecosystem")["label"].mean()).values)
proxy = {f: {e: float(x) for e, x in df.groupby("ecosystem")[f].mean().items()}
         for f in SEL_B if (df.groupby("ecosystem")[f].std().fillna(0) == 0).any()}
print(f"\nComposition par ecosysteme :\n{comp.to_string()}")
print(f"AUC d'un modele qui n'utilise QUE l'ecosysteme : {auc_eco:.4f}")
print(f"   -> comparer a l'AUC du modele ({TB.set_index('model').loc[BEST,'test_auc_roc']:.4f}).")
print(f"Features constantes dans un ecosysteme (proxies) : {list(proxy)}")
auc_m_ = TB.set_index("model").loc[BEST, "test_auc_roc"]
share = (auc_eco - 0.5)/(auc_m_ - 0.5) if auc_m_ > 0.5 else np.nan
ratios = df.groupby("ecosystem")["label"].agg(["mean","size"])
print(f"Part du pouvoir discriminant au-dela du hasard attribuable a l'ecosysteme : {share:.1%}")
print(f"Features constantes dans un ecosysteme : {len(proxy)} sur {len(SEL_B)}")
print("Ratio malveillant par ecosysteme :\n" + ratios.to_string())
if share > 0.5:
    print("!! Plus de la moitie du signal vient de l'ecosysteme. Soit equilibrer la "
          "composition par ecosysteme, soit restreindre le papier a NPM.")
R["ecosystem_confound"] = {"composition": comp.to_dict(), "auc_ecosystem_only": float(auc_eco),
                           "auc_model": float(auc_m_),
                           "share_of_discriminative_power": float(share),
                           "n_proxy_features": len(proxy), "n_selected": len(SEL_B),
                           "malicious_ratio_by_ecosystem": ratios.to_dict(),
                           "single_ecosystem_features": proxy}

def fit_eval(tr, te, feats=None):
    feats = feats or SEL_B
    p, _ = make_prep(tr, ("vman_","pit_"))
    m = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, l2_leaf_reg=3,
                           verbose=0, random_state=SEED, auto_class_weights="Balanced")
    m.fit(p(tr)[feats], tr.label.values); s = m.predict_proba(p(te)[feats])[:,1]
    out = {"auc_roc": roc_auc_score(te.label.values, s),
           "au_pr": average_precision_score(te.label.values, s),
           "au_pr_baseline": float(te.label.mean()),
           "n_train": len(tr), "n_test": len(te), "n_test_malicious": int(te.label.sum())}
    if out["n_test_malicious"] < 50: out["warning"] = "trop peu de positifs pour etre interpretable"
    return out
cross = {}
for a, b in [("npm","pypi"), ("pypi","npm")]:
    A_, B_ = df[df.ecosystem==a], df[df.ecosystem==b]
    if len(A_) and len(B_) and A_.label.nunique()>1 and B_.label.nunique()>1:
        cross[f"{a}->{b}"] = fit_eval(A_, B_)
cut = df.t_p.median()
tr_t, te_t = df[df.t_p < cut], df[df.t_p >= cut]
oot = (fit_eval(tr_t, te_t) if len(tr_t) and len(te_t)
       and tr_t.label.nunique()>1 and te_t.label.nunique()>1 else None)
span_days = (df.t_p.max() - df.t_p.min()).days
print("\nCross-ecosysteme :", json.dumps(cross, indent=2))
print(f"Hors-temps (cutoff {cut.date()}, etendue totale {span_days} j) :", oot)
if span_days < 365: print("   !! etendue < 1 an : ce test ne mesure aucune derive, ne pas le publier.")
R["generalisation"] = {"cross_ecosystem": cross, "out_of_time": oot,
                       "cutoff": str(cut.date()), "data_span_days": int(span_days)}

# =============================================================================
# SUITE D'ABLATIONS — R3.2, R3.3, et ce que les relecteurs n'ont pas demande
# mais qui decide de la contribution du papier.
#
# Les ablations de la version precedente etaient muettes par construction :
#   - l'ablation de variance tournait APRES le pre-filtre ecosysteme, sur 40
#     features dont aucune n'a une variance < 0.01 : l'etape n'avait rien a
#     retirer, donc son retrait ne changeait rien ;
#   - l'ablation d'ordre etait ecrasee par l'etape budget, qui tronque chaque
#     chemin aux 18 meilleures : les differences d'ordre disparaissaient a la
#     derniere etape.
# Les deux sont corrigees ici, et six experiences sont ajoutees.
# =============================================================================
ABL = {}

def ev(feats, model="cat"):
    feats = [f for f in feats if f in Xtr.columns]
    if len(feats) == 0 or len(np.unique(y_tr)) < 2:
        return {"k": 0, "auc_roc": np.nan, "au_pr": np.nan}
    if model == "cat":
        m = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, l2_leaf_reg=3,
                               verbose=0, random_state=SEED, auto_class_weights="Balanced")
    else:
        m = RandomForestClassifier(n_estimators=200, max_depth=20, min_samples_split=5,
                                   max_features="sqrt", class_weight="balanced",
                                   random_state=SEED, n_jobs=-1)
    m.fit(Xtr[feats], y_tr); s = m.predict_proba(Xte[feats])[:, 1]
    return {"k": len(feats), "auc_roc": roc_auc_score(y_te, s),
            "au_pr": average_precision_score(y_te, s),
            "au_pr_baseline": float(np.mean(y_te))}

def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if (a | b) else np.nan

CI_W = ci[BEST]["au_pr_ci95"][1] - ci[BEST]["au_pr_ci95"][0]
print(f"\nLargeur de l'IC95 sur l'AU-PR : {CI_W:.4f}  "
      f"(tout ecart inferieur n'est pas distinguable du bruit)")
ABL["au_pr_ci_width"] = float(CI_W)

# ─────────────────────────────────────────────────────────────────────────────
# A. COURBE CUMULATIVE PAR ETAPE — la reduction coute-t-elle quelque chose ?
# ─────────────────────────────────────────────────────────────────────────────
print("\n[A] Courbe cumulative par etape")
STAGES_SEQ = ["variance", "mi", "boruta", "rfe", "corr", "budget"]
cum, prefix = [], []
cum.append({"stage": "initial", **ev(list(Xtr.columns))})
for st in STAGES_SEQ:
    prefix.append(st)
    feats, lg = select(Xtr, y_tr, stages=tuple(prefix), verbose=False)
    cum.append({"stage": st, **ev(feats)})
CUM = pd.DataFrame(cum)
print(CUM[["stage", "k", "auc_roc", "au_pr"]].to_string(index=False))
d_full = CUM.iloc[0]["au_pr"] - CUM.iloc[-1]["au_pr"]
print(f"   AU-PR perdue entre l'espace complet et 18 features : {d_full:+.4f} "
      f"({'dans le bruit' if abs(d_full) < CI_W else 'superieure au bruit'})")
ABL["cumulative"] = CUM.to_dict("records")
ABL["cost_of_reduction_au_pr"] = float(d_full)

# ─────────────────────────────────────────────────────────────────────────────
# B. LEAVE-ONE-STAGE-OUT — contribution marginale de chaque etape
# ─────────────────────────────────────────────────────────────────────────────
print("\n[B] Leave-one-stage-out")
full_feats, _ = select(Xtr, y_tr, stages=tuple(STAGES_SEQ), verbose=False)
base = ev(full_feats)
rows = [{"removed": "none (full pipeline)", **base, "overlap_with_full": 1.0,
         "delta_au_pr": 0.0}]
for st in STAGES_SEQ:
    sub = tuple(s for s in STAGES_SEQ if s != st)
    f_, _ = select(Xtr, y_tr, stages=sub, verbose=False)
    e = ev(f_)
    rows.append({"removed": st, **e,
                 "overlap_with_full": jaccard(f_, full_feats),
                 "delta_au_pr": e["au_pr"] - base["au_pr"]})
LOSO = pd.DataFrame(rows)
print(LOSO[["removed", "k", "auc_roc", "au_pr", "delta_au_pr",
            "overlap_with_full"]].to_string(index=False))
ABL["leave_one_stage_out"] = LOSO.to_dict("records")

# ─────────────────────────────────────────────────────────────────────────────
# C. VARIANCE : teste sur l'espace COMPLET, pas apres le pre-filtre
#    (correction du defaut de conception de la version precedente)
# ─────────────────────────────────────────────────────────────────────────────
print("\n[C] Ablation du seuil de variance, sur l'espace point-in-time complet")
ECO_SAVE = list(ECO_PROXIES)
ECO_PROXIES = []                                  # on rouvre l'espace complet
prep_full, F_full = make_prep(train, ("vman_", "pit_"))
Xtr_f, Xte_f = prep_full(train), prep_full(test)

def ev_on(Xa, Xb, feats):
    feats = [f for f in feats if f in Xa.columns]
    m = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, l2_leaf_reg=3,
                           verbose=0, random_state=SEED, auto_class_weights="Balanced")
    m.fit(Xa[feats], y_tr); s = m.predict_proba(Xb[feats])[:, 1]
    return {"k": len(feats), "auc_roc": roc_auc_score(y_te, s),
            "au_pr": average_precision_score(y_te, s)}

with_v, log_w = select(Xtr_f, y_tr, stages=("variance","mi","boruta","rfe","corr","budget"),
                       verbose=False)
no_v,  log_n = select(Xtr_f, y_tr, stages=("mi","boruta","rfe","corr","budget"),
                      verbose=False)
VAR_ABL = {"space": int(Xtr_f.shape[1]),
           "with_variance":    {**ev_on(Xtr_f, Xte_f, with_v), "log": log_w},
           "without_variance": {**ev_on(Xtr_f, Xte_f, no_v),  "log": log_n},
           "jaccard": jaccard(with_v, no_v),
           "recovered_without_variance": sorted(set(no_v) - set(with_v)),
           "lost_without_variance": sorted(set(with_v) - set(no_v))}
print(json.dumps({k: v for k, v in VAR_ABL.items() if k != "recovered_without_variance"},
                 indent=2, default=str)[:900])
print("   features remontant sans le seuil de variance :",
      VAR_ABL["recovered_without_variance"][:10])
ABL["variance_on_full_space"] = VAR_ABL
ECO_PROXIES = ECO_SAVE                            # on restaure le pre-filtre

# ─────────────────────────────────────────────────────────────────────────────
# D. ORDRE : mesure AVANT le budget, avec recouvrement des ensembles
# ─────────────────────────────────────────────────────────────────────────────
print("\n[D] Ablation d'ordre, mesuree AVANT l'etape budget")
ORD = [("variance","mi","boruta","rfe","corr"),
       ("corr","variance","mi","boruta","rfe"),
       ("variance","mi","corr","boruta","rfe"),
       ("variance","corr","mi","boruta","rfe"),
       ("mi","variance","boruta","corr","rfe"),
       ("boruta","variance","mi","rfe","corr")]
ref_set = None; rows = []
for o in ORD:
    f_, lg = select(Xtr, y_tr, stages=o, verbose=False)
    if ref_set is None: ref_set = f_
    rows.append({"order": " -> ".join(o), **ev(f_),
                 "jaccard_with_first": jaccard(f_, ref_set)})
AO_PRE = pd.DataFrame(rows).sort_values("au_pr", ascending=False)
print(AO_PRE[["order", "k", "auc_roc", "au_pr", "jaccard_with_first"]].to_string(index=False))
spread_pre = float(AO_PRE["au_pr"].max() - AO_PRE["au_pr"].min())
print(f"   ecart max AU-PR = {spread_pre:.4f} | IC95 = {CI_W:.4f} -> "
      f"{'aucun ordre distinguable' if spread_pre < CI_W else 'ecart superieur au bruit'}")
print(f"   recouvrement moyen des ensembles selectionnes : "
      f"{AO_PRE['jaccard_with_first'].mean():.3f}")
ABL["order_before_budget"] = {"table": AO_PRE.to_dict("records"),
                              "spread_au_pr": spread_pre,
                              "distinguishable": bool(spread_pre >= CI_W),
                              "mean_jaccard": float(AO_PRE["jaccard_with_first"].mean())}

# meme mesure APRES budget, pour montrer que le budget homogeneise
rows = []
for o in ORD:
    f_, _ = select(Xtr, y_tr, stages=o + ("budget",), verbose=False)
    rows.append({"order": " -> ".join(o) + " -> budget", **ev(f_)})
AO_POST = pd.DataFrame(rows).sort_values("au_pr", ascending=False)
spread_post = float(AO_POST["au_pr"].max() - AO_POST["au_pr"].min())
print(f"   apres budget : ecart max = {spread_post:.4f} "
      f"(le budget homogeneise les chemins)")
ABL["order_after_budget"] = {"table": AO_POST.to_dict("records"),
                             "spread_au_pr": spread_post}

# ─────────────────────────────────────────────────────────────────────────────
# E. COMPARAISON A BUDGET EGAL — l'experience decisive
#    Le pipeline hybride bat-il un selecteur simple a k = 18 ?
# ─────────────────────────────────────────────────────────────────────────────
print("\n[E] Comparaison a budget egal (k = 18)")
K = CFG["final_feature_budget"]
cands = {}

cands["Hybrid pipeline (ours)"] = SEL_B

mi_ = mutual_info_classif(Xtr, y_tr, random_state=SEED)
cands["Mutual information top-k"] = list(pd.Series(mi_, index=Xtr.columns)
                                         .sort_values(ascending=False).head(K).index)

rf_ = RandomForestClassifier(n_estimators=300, max_depth=None, class_weight="balanced",
                             random_state=SEED, n_jobs=-1).fit(Xtr, y_tr)
cands["RF importance top-k"] = list(pd.Series(rf_.feature_importances_, index=Xtr.columns)
                                    .sort_values(ascending=False).head(K).index)

cb_ = CatBoostClassifier(iterations=300, learning_rate=0.1, depth=6, verbose=0,
                         random_state=SEED, auto_class_weights="Balanced").fit(Xtr, y_tr)
cands["CatBoost importance top-k"] = list(pd.Series(cb_.get_feature_importance(),
                                                    index=Xtr.columns)
                                          .sort_values(ascending=False).head(K).index)

from sklearn.linear_model import LogisticRegression as _LR
l1_ = _LR(penalty="l1", solver="liblinear", C=0.1, class_weight="balanced",
          max_iter=2000).fit(Xtr, y_tr)
cands["L1 logistic top-k"] = list(pd.Series(np.abs(l1_.coef_[0]), index=Xtr.columns)
                                  .sort_values(ascending=False).head(K).index)

rows = [{"selector": nm, **ev(fs), "jaccard_with_ours": jaccard(fs, SEL_B)}
        for nm, fs in cands.items()]

# tirage aleatoire : le plancher
rng_r = np.random.default_rng(SEED); rnd = []
for _ in range(CFG["n_random_baseline"]):
    fs = list(rng_r.choice(Xtr.columns, size=min(K, Xtr.shape[1]), replace=False))
    rnd.append((ev(fs)["auc_roc"], ev(fs)["au_pr"], jaccard(fs, SEL_B)))
rnd = np.array(rnd)
rows.append({"selector": f"Random-{K} (median of {CFG['n_random_baseline']})",
             "k": K, "auc_roc": float(np.median(rnd[:, 0])),
             "au_pr": float(np.median(rnd[:, 1])),
             "au_pr_baseline": float(np.mean(y_te)),
             "jaccard_with_ours": float(np.median(rnd[:, 2]))})

# PCA a dimension egale
from sklearn.decomposition import PCA as _PCA
pca = _PCA(n_components=min(K, Xtr.shape[1]), random_state=SEED).fit(Xtr)
mp = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, verbose=0,
                        random_state=SEED, auto_class_weights="Balanced")
mp.fit(pca.transform(Xtr), y_tr); sp = mp.predict_proba(pca.transform(Xte))[:, 1]
rows.append({"selector": f"PCA-{K} components", "k": K,
             "auc_roc": roc_auc_score(y_te, sp),
             "au_pr": average_precision_score(y_te, sp),
             "au_pr_baseline": float(np.mean(y_te)), "jaccard_with_ours": np.nan})

MATCHED = pd.DataFrame(rows).sort_values("au_pr", ascending=False)
print(MATCHED[["selector", "k", "auc_roc", "au_pr", "jaccard_with_ours"]].to_string(index=False))
ours = MATCHED.loc[MATCHED.selector.str.contains("Hybrid"), "au_pr"].iloc[0]
best_other = MATCHED.loc[~MATCHED.selector.str.contains("Hybrid"), "au_pr"].max()
gain = ours - best_other
print(f"   ecart avec le meilleur selecteur alternatif : {gain:+.4f} | IC95 = {CI_W:.4f}")
print("   -> " + ("l'avantage du pipeline hybride est superieur au bruit"
                  if gain >= CI_W else
                  "l'avantage du pipeline hybride N'EST PAS distinguable du bruit ; "
                  "la contribution doit etre presentee comme empirique "
                  "(compressibilite de l'espace) et non methodologique"))
ABL["matched_budget"] = {"table": MATCHED.to_dict("records"),
                         "gain_over_best_alternative": float(gain),
                         "gain_exceeds_ci": bool(gain >= CI_W)}

# ─────────────────────────────────────────────────────────────────────────────
# F. COURBE DE BUDGET — 18 est-il au coude ?
# ─────────────────────────────────────────────────────────────────────────────
print("\n[F] Courbe performance / nombre de features")
rows = []
for k in CFG["budget_sweep"]:
    saved = CFG["final_feature_budget"]; CFG["final_feature_budget"] = k
    f_, _ = select(Xtr, y_tr, stages=tuple(STAGES_SEQ), verbose=False)
    CFG["final_feature_budget"] = saved
    rows.append({"budget": k, **ev(f_)})
rows.append({"budget": Xtr.shape[1], **ev(list(Xtr.columns))})
SWEEP = pd.DataFrame(rows)
print(SWEEP[["budget", "k", "auc_roc", "au_pr"]].to_string(index=False))
plt.figure(figsize=(6.2, 4.0))
plt.plot(SWEEP["k"], SWEEP["au_pr"], "o-", color="#1F4E9C", label="AU-PR")
plt.plot(SWEEP["k"], SWEEP["auc_roc"], "s--", color="#E8973A", label="AUC-ROC")
plt.axvline(CFG["final_feature_budget"], color="grey", ls=":",
            label=f"budget = {CFG['final_feature_budget']}")
plt.axhline(float(np.mean(y_te)), color="#999999", ls="-.", lw=0.9,
            label="AU-PR baseline")
plt.xscale("log"); plt.xlabel("Number of features"); plt.ylabel("Score")
plt.title("Performance versus feature-set size (test partition)")
plt.legend(fontsize=8); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(f"{OUT}/figure6_budget_sweep.png", dpi=600); plt.show()
ABL["budget_sweep"] = SWEEP.to_dict("records")

# ─────────────────────────────────────────────────────────────────────────────
# G. STABILITE DE L'ENSEMBLE SELECTIONNE (Jaccard, bootstrap groupe)
# ─────────────────────────────────────────────────────────────────────────────
print("\n[G] Stabilite de l'ensemble selectionne")
rng_s = np.random.default_rng(SEED)
g_arr = train.group_id.values; uq_g = np.unique(g_arr); sets = []
for _ in range(CFG["n_stability_runs"]):
    keep = set(rng_s.choice(uq_g, size=len(uq_g), replace=True))
    mk = np.array([g in keep for g in g_arr])
    if len(np.unique(y_tr[mk])) < 2: continue
    f_, _ = select(Xtr[mk], y_tr[mk], stages=tuple(STAGES_SEQ), verbose=False)
    sets.append(f_)
pairs = [jaccard(a, b) for a, b in itertools.combinations(sets, 2)]
freq = pd.Series([f for s in sets for f in s]).value_counts() / max(len(sets), 1)
print(f"   Jaccard moyen entre {len(sets)} ensembles : {np.mean(pairs):.3f} "
      f"(mediane {np.median(pairs):.3f})")
print(f"   features retenues dans >= 80% des tirages : "
      f"{int((freq >= 0.8).sum())} sur {CFG['final_feature_budget']}")
ABL["selection_stability"] = {"n_runs": len(sets),
                              "mean_jaccard": float(np.mean(pairs)) if pairs else np.nan,
                              "median_jaccard": float(np.median(pairs)) if pairs else np.nan,
                              "selection_frequency": freq.round(3).to_dict(),
                              "n_features_selected_80pct": int((freq >= 0.8).sum())}

# ─────────────────────────────────────────────────────────────────────────────
# H. ABLATION PAR FAMILLE DE METADONNEES — ou est le signal ?
# ─────────────────────────────────────────────────────────────────────────────
print("\n[H] Ablation par famille de metadonnees")
FAMILIES = {
    "authorship":     [f for f in SEL_B if any(k in f for k in
                       ["author", "maintainer", "email"])],
    "release_history":[f for f in SEL_B if any(k in f for k in
                       ["prior_releases", "releases_30d", "releases_365d",
                        "days_since", "gap_days"])],
    "description":    [f for f in SEL_B if "description" in f or "keyword" in f],
    "versioning":     [f for f in SEL_B if "ver_" in f or "version" in f],
    "dependencies":   [f for f in SEL_B if "depend" in f],
    "identity":       [f for f in SEL_B if any(k in f for k in
                       ["name_", "repository", "homepage", "license"])],
}
rows = [{"removed_family": "none", "n_removed": 0, **ev(SEL_B), "delta_au_pr": 0.0}]
b = ev(SEL_B)
for nm, fs in FAMILIES.items():
    if not fs: continue
    rest = [f for f in SEL_B if f not in fs]
    if not rest: continue
    e = ev(rest)
    rows.append({"removed_family": nm, "n_removed": len(fs), **e,
                 "delta_au_pr": e["au_pr"] - b["au_pr"]})
FAM = pd.DataFrame(rows).sort_values("delta_au_pr")
print(FAM[["removed_family", "n_removed", "k", "auc_roc", "au_pr",
           "delta_au_pr"]].to_string(index=False))
ABL["family_ablation"] = {"membership": {k: v for k, v in FAMILIES.items()},
                          "table": FAM.to_dict("records")}

# ─────────────────────────────────────────────────────────────────────────────
# I. PRE-FILTRE ECOSYSTEME : avec / sans, controle de sensibilite
# ─────────────────────────────────────────────────────────────────────────────
print("\n[I] Pre-filtre ecosysteme : controle de sensibilite")
sel_nofilter, log_nf = select(Xtr_f, y_tr, stages=tuple(STAGES_SEQ), verbose=False)
e_nf = ev_on(Xtr_f, Xte_f, sel_nofilter)
n_proxy_nf = len([f for f in sel_nofilter
                  if (df.groupby("ecosystem")[f].std().fillna(0) == 0).any()])
ECO_FILTER = {"with_prefilter":   {**ev(SEL_B), "n_proxy_in_selection": len(proxy)},
              "without_prefilter":{**e_nf, "n_proxy_in_selection": n_proxy_nf},
              "jaccard": jaccard(sel_nofilter, SEL_B)}
print(json.dumps(ECO_FILTER, indent=2, default=str))
print("   -> l'ecart mesure le cout, en performance apparente, du retrait du confondant")
ABL["ecosystem_prefilter_sensitivity"] = ECO_FILTER

# ─────────────────────────────────────────────────────────────────────────────
# retro-compatibilite avec le reste de la cellule
# ─────────────────────────────────────────────────────────────────────────────
abl_var = {"with_variance": VAR_ABL["with_variance"],
           "without_variance": VAR_ABL["without_variance"],
           "overlap": VAR_ABL["jaccard"],
           "recovered": VAR_ABL["recovered_without_variance"]}
AO = AO_PRE.copy()
spread = spread_pre
R["ablations"] = ABL


FREE = [f for f in SEL_B if any(k in f for k in
        ["description","summary","keyword","license","depend","repository","homepage",
         "ver_len","ver_n_components","ver_entropy","classifier","requires_python","name_"])]
COSTLY = [f for f in SEL_B if any(k in f for k in
          ["prior_releases","releases_30d","releases_365d","days_since","gap_days"])]
FIXED = [f for f in SEL_B if f not in FREE + COSTLY]
med = Xtr_s.loc[y_tr == 0].median(); rows_ev = []
for lab, fs in [("free", FREE), ("free+costly", FREE + COSTLY)]:
    for bgt in range(0, len(fs)+1):
        Xa = Xte_s.copy()
        for f in fs[:bgt]: Xa.loc[y_te == 1, f] = med[f]
        s = mbest.predict_proba(Xa)[:,1]
        rows_ev.append({"feature_set": lab, "budget": bgt,
                        "auc_roc": roc_auc_score(y_te, s),
                        "au_pr": average_precision_score(y_te, s),
                        "recall_at_threshold": recall_score(y_te, (s>=thr).astype(int),
                                                            pos_label=1, zero_division=0)})
EV = pd.DataFrame(rows_ev); print("\n", EV.head(20).to_string(index=False))
R["evasion"] = {"free": FREE, "costly": COSTLY, "fixed": FIXED, "curve": EV.to_dict("records")}

sv = shap.TreeExplainer(fitB["CatBoost"]).shap_values(Xte_s)   # jamais sur le Stacking
if isinstance(sv, list): sv = sv[1]
shap.summary_plot(sv, Xte_s, show=False)
plt.title("SHAP - CatBoost (protocole B)"); plt.tight_layout()
plt.savefig(f"{OUT}/figure4_shap_catboost.png", dpi=300, bbox_inches="tight"); plt.show()
GLOB = pd.DataFrame({"feature": SEL_B, "mean_abs_shap": np.abs(sv).mean(0)}
                    ).sort_values("mean_abs_shap", ascending=False)
print(GLOB.to_string(index=False))
rng = np.random.default_rng(SEED); g_tr = train.group_id.values; uq = np.unique(g_tr); RK = []
for _ in range(10):
    sel_g = set(rng.choice(uq, size=len(uq), replace=True))
    mk = np.array([g in sel_g for g in g_tr])
    if len(np.unique(y_tr[mk])) < 2: continue
    mm = CatBoostClassifier(iterations=300, learning_rate=0.1, depth=6, verbose=0,
                            random_state=SEED, auto_class_weights="Balanced").fit(Xtr_s[mk], y_tr[mk])
    v = shap.TreeExplainer(mm).shap_values(Xte_s); v = v[1] if isinstance(v, list) else v
    RK.append(pd.Series(np.abs(v).mean(0), index=SEL_B).rank(ascending=False))
RKdf = pd.concat(RK, axis=1)
stab = float(np.mean([spearmanr(RKdf.iloc[:,i], RKdf.iloc[:,j]).correlation
                      for i, j in itertools.combinations(range(RKdf.shape[1]), 2)]))
pi = permutation_importance(fitB["CatBoost"], Xte_s, y_te, n_repeats=10,
                            random_state=SEED, scoring="average_precision")
agree = float(spearmanr(GLOB.set_index("feature").loc[SEL_B,"mean_abs_shap"],
                        pi.importances_mean).correlation)
print(f"Stabilite des rangs SHAP : {stab:.3f} | accord SHAP/permutation : {agree:.3f}")
sb = scoB[BEST]["test"]; pb = (sb >= thr).astype(int); loc = {"FP": [], "FN": []}
for key, idxs in [("FP", np.where((pb==1)&(y_te==0))[0][:5]),
                  ("FN", np.where((pb==0)&(y_te==1))[0][:5])]:
    for i in idxs:
        c = pd.Series(sv[i], index=SEL_B).sort_values(key=abs, ascending=False)
        loc[key].append({"package": str(test.iloc[i]["package"]),
                         "version": str(test.iloc[i]["version"]),
                         "score": float(sb[i]), "top": c.head(5).round(4).to_dict()})
R["shap"] = {"model_explained": "CatBoost", "global": GLOB.to_dict("records"),
             "rank_stability_spearman": stab,
             "agreement_permutation_spearman": agree, "local_examples": loc}


# =============================================================================
# LATENCE INSTRUMENTEE                                                  [R1.7]
# =============================================================================
import time as _time
def measure_latency(model, X, n_rep=30, batch=1):
    lat = []
    for i in range(n_rep):
        xb = X.iloc[[i % len(X)]] if batch == 1 else X.iloc[:min(batch, len(X))]
        t0 = _time.perf_counter(); model.predict_proba(xb); lat.append(_time.perf_counter()-t0)
    a = np.array(lat)*1000
    return {"median_ms": float(np.median(a)),
            "iqr_ms": float(np.percentile(a,75)-np.percentile(a,25)),
            "n_repetitions": n_rep, "batch_size": batch}

sample = test.sample(min(20, len(test)), random_state=SEED)
t_fetch = []
for _, r_ in sample.iterrows():
    t0 = _time.perf_counter()
    (npm_features if r_["ecosystem"] == "npm" else pypi_features)(r_["package"], r_["version"])
    t_fetch.append((_time.perf_counter()-t0)*1000)
LAT = {"inference": {str(b): measure_latency(mbest, Xte_s, batch=b) for b in (1, 32, 256)},
       "registry_fetch_and_feature_extraction": {
           "median_ms": float(np.median(t_fetch)),
           "iqr_ms": float(np.percentile(t_fetch,75)-np.percentile(t_fetch,25)),
           "n_repetitions": len(t_fetch)},
       "note": "la latence bout-en-bout est dominee par l'acces au registre, pas par "
               "l'inference ; ne pas en conclure une capacite temps reel"}
print("\nLatence :", json.dumps(LAT, indent=2))
R["latency"] = LAT

# =============================================================================
# CALIBRATION : Brier, ECE, diagramme de fiabilite                      [R1.7]
# =============================================================================
from sklearn.calibration import calibration_curve
def ece(y, s, bins=10):
    edges = np.linspace(0, 1, bins+1); e, n = 0.0, len(y)
    for i in range(bins):
        m_ = (s > edges[i]) & (s <= edges[i+1])
        if m_.sum() == 0: continue
        e += (m_.sum()/n) * abs(y[m_].mean() - s[m_].mean())
    return float(e)
s_best = scoB[BEST]["test"]
frac, mean_pred = calibration_curve(y_te, s_best, n_bins=10, strategy="quantile")
plt.figure(figsize=(5.5, 5))
plt.plot(mean_pred, frac, "o-", label=BEST)
plt.plot([0,1], [0,1], "k--", alpha=.5, label="parfaitement calibre")
plt.xlabel("Probabilite predite moyenne"); plt.ylabel("Fraction observee de positifs")
plt.title(f"Reliability diagram - {BEST} (protocole B)"); plt.legend(); plt.tight_layout()
plt.savefig(f"{OUT}/figure5_calibration.png", dpi=300); plt.show()
CAL = {"brier": float(brier_score_loss(y_te, s_best)), "ece_10bins": ece(y_te, s_best),
       "curve": [{"mean_predicted": float(a), "observed_fraction": float(b)}
                 for a, b in zip(mean_pred, frac)]}
print("Calibration :", json.dumps({k: v for k, v in CAL.items() if k != "curve"}, indent=2))
R["calibration"] = CAL

# =============================================================================
# ROBUSTESSE : bruit d'etiquettes et corruption, AVEC repetitions       [R1.9]
# =============================================================================
# La Section 3.5 du manuscrit rapporte 0.962 et 0.941 sans protocole, sans
# definition de perturbation et sans repetition. Definitions explicites ici.
N_REP = 10
def robustness_label_noise(rate=0.10, n_rep=N_REP):
    rng_ = np.random.default_rng(SEED); vals = []
    for _ in range(n_rep):
        yn = y_tr.copy()
        idx = rng_.choice(len(yn), int(rate*len(yn)), replace=False)
        yn[idx] = 1 - yn[idx]
        m_ = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, verbose=0,
                                random_state=SEED, auto_class_weights="Balanced")
        m_.fit(Xtr_s, yn); s_ = m_.predict_proba(Xte_s)[:,1]
        vals.append((roc_auc_score(y_te, s_), average_precision_score(y_te, s_)))
    a = np.array(vals)
    return {"definition": f"{int(rate*100)}% des etiquettes d'ENTRAINEMENT inversees uniformement",
            "n_repetitions": n_rep,
            "auc_roc_median": float(np.median(a[:,0])),
            "auc_roc_iqr": float(np.percentile(a[:,0],75)-np.percentile(a[:,0],25)),
            "au_pr_median": float(np.median(a[:,1])),
            "au_pr_iqr": float(np.percentile(a[:,1],75)-np.percentile(a[:,1],25))}

def robustness_corruption(rate=0.20, n_rep=N_REP):
    rng_ = np.random.default_rng(SEED); vals = []
    for _ in range(n_rep):
        Xc = Xte_s.copy()
        mask = rng_.random(Xc.shape) < rate
        for j, col in enumerate(Xc.columns):
            k_ = int(mask[:, j].sum())
            if k_: Xc.loc[mask[:, j], col] = rng_.choice(Xtr_s[col].values, k_, replace=True)
        s_ = mbest.predict_proba(Xc)[:,1]
        vals.append((roc_auc_score(y_te, s_), average_precision_score(y_te, s_)))
    a = np.array(vals)
    return {"definition": f"{int(rate*100)}% des valeurs de TEST remplacees par un tirage dans "
                          f"la marginale d'entrainement de la meme feature",
            "n_repetitions": n_rep,
            "auc_roc_median": float(np.median(a[:,0])),
            "auc_roc_iqr": float(np.percentile(a[:,0],75)-np.percentile(a[:,0],25)),
            "au_pr_median": float(np.median(a[:,1])),
            "au_pr_iqr": float(np.percentile(a[:,1],75)-np.percentile(a[:,1],25))}

ROB = {"label_noise_10pct": robustness_label_noise(),
       "feature_corruption_20pct": robustness_corruption()}
print("\nRobustesse :", json.dumps(ROB, indent=2))
R["robustness"] = ROB

# =============================================================================
# CONTROLE APPARIE SUR LA MATURITE                                      [R1.4]
# =============================================================================
AGE, REL = "pit_days_since_first", "pit_n_prior_releases"
MATCH = {"warning": "features d'age/cadence absentes"}
if AGE in df.columns and REL in df.columns:
    mal_ = df[df.label == 1]; ben_ = df[df.label == 0]
    used, pairs = set(), []
    for _, m_ in mal_.iterrows():
        c_ = ben_[(~ben_.index.isin(used))
                  & (ben_[AGE].sub(m_[AGE]).abs() <= 90)
                  & (ben_[REL].sub(m_[REL]).abs() <= 2)]
        if len(c_):
            j = c_.index[0]; used.add(j); pairs += [m_.name, j]
    dm = df.loc[pairs]
    print(f"\nEchantillon apparie : {len(dm)} versions ({len(dm)//2} paires), "
          f"{dm.label.mean():.1%} malveillant")
    if len(dm) > 200 and dm.label.nunique() > 1 and dm.group_id.nunique() > 10:
        i1_, i2_ = next(GroupShuffleSplit(1, test_size=CFG["test_frac"], random_state=SEED)
                        .split(dm, dm.label, groups=dm.group_id))
        MATCH = fit_eval(dm.iloc[i1_], dm.iloc[i2_])
        MATCH["unmatched_reference"] = {
            "auc_roc": float(TB.set_index("model").loc[BEST, "test_auc_roc"]),
            "au_pr": float(TB.set_index("model").loc[BEST, "test_au_pr"])}
        MATCH["n_pairs"] = len(dm)//2
        print("Apparie :", json.dumps(MATCH, indent=2))
        print("-> l'ecart avec la reference non appariee mesure le confondant de maturite.")
    else:
        MATCH = {"warning": "echantillon apparie trop petit", "n_pairs": len(dm)//2}
        print(MATCH)
R["maturity_matched"] = MATCH

# =============================================================================
# EXPORT
# =============================================================================
def clean(o):
    if isinstance(o, dict):  return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, list):  return [clean(v) for v in o]
    if isinstance(o, np.integer):  return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.str_):     return str(o)
    return o
R["generated_at"] = datetime.now(timezone.utc).isoformat()
json.dump(clean(R), open(f"{OUT}/revision_results.json","w"), indent=2, default=str)
CMP.to_csv(f"{OUT}/table4_protocols_A_B.csv", index=False)
GLOB.to_csv(f"{OUT}/table3_feature_importance.csv", index=False)
PT.to_csv(f"{OUT}/table6_operational_prevalence.csv", index=False)
AO.to_csv(f"{OUT}/table7_ablation_order.csv", index=False)
EV.to_csv(f"{OUT}/table8_evasion.csv", index=False)
pd.DataFrame([{"stage": k, "protocol_A": LOG_A.get(k), "protocol_B": LOG_B.get(k)}
              for k in ["initial","variance","mutual_info","boruta","rfe","corr_prune","budget"]]
             ).to_csv(f"{OUT}/table2_selection_stages.csv", index=False)
pd.concat([p[["package","version","ecosystem","group_id"]].assign(split=n)
           for p, n in [(train,"train"), (val,"val"), (test,"test")]]
          ).to_csv(f"{OUT}/partition_assignment.csv", index=False)
df.to_csv(f"{OUT}/dataset_versions_point_in_time.csv", index=False)
pd.DataFrame([{"experiment": k, **v} for k, v in ROB.items()]
             ).to_csv(f"{OUT}/table9_robustness.csv", index=False)
pd.DataFrame(CAL["curve"]).to_csv(f"{OUT}/figure5_calibration_data.csv", index=False)
CUM.to_csv(f"{OUT}/table10_cumulative_stages.csv", index=False)
LOSO.to_csv(f"{OUT}/table11_leave_one_stage_out.csv", index=False)
MATCHED.to_csv(f"{OUT}/table12_matched_budget_selectors.csv", index=False)
SWEEP.to_csv(f"{OUT}/figure6_budget_sweep_data.csv", index=False)
FAM.to_csv(f"{OUT}/table13_family_ablation.csv", index=False)
AO_PRE.to_csv(f"{OUT}/table7_ablation_order_pre_budget.csv", index=False)
cm = confusion_matrix(y_te, (scoB[BEST]["test"] >= thr).astype(int), labels=[0,1])
assert cm.sum() == len(y_te), "Figure 3 incoherente avec la Table 1"
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Benign","Malicious"], yticklabels=["Benign","Malicious"])
plt.title(f"Confusion matrix - {BEST}, protocol B (n={cm.sum()})")
plt.ylabel("True label"); plt.xlabel("Predicted"); plt.tight_layout()
plt.savefig(f"{OUT}/figure3_confusion_matrix.png", dpi=300); plt.show()

y = R["collection"].get("yield")
print(f"""
=====================================================================
CORRESPONDANCE MANUSCRIT
Table 1  <- R['dataset'] + R['protocol_B']['split']   (ne pas ecrire "balanced")
Table 2  <- table2_selection_stages.csv               (A et B cote a cote)
Table 3  <- table3_feature_importance.csv
Table 4  <- table4_protocols_A_B.csv
Table S3 <- supplementary_S3_feature_inventory.csv    (inventaire complet, R2.1)
Table S5 <- R['hyperparameters']
Figure 2 <- effectifs ENTIERS par categorie, pas des pourcentages
Figure 3 <- figure3_confusion_matrix.png, somme = {len(y_te)}
Figure 4 <- figure4_shap_catboost.png   (legender : CatBoost, protocole B)
Sec 3.2  <- table7_ablation_order.csv + R['ablations']['variance']
Sec 3.5  <- table8_evasion.csv + R['generalisation'] + R['ecosystem_confound']
Sec 3.6  <- table6_operational_prevalence.csv  (ligne 'upper95')
---------------------------------------------------------------------
Espace de features : {R['feature_space']['total']} au total
   vman {R['feature_space']['vman']} | pit {R['feature_space']['pit']} | cur {R['feature_space']['cur']}
Protocole A : {LOG_A['initial']} -> {len(SEL_A)} | accuracy {TA.set_index('model').loc[BEST_A,'test_accuracy']:.4f}
Protocole B : {LOG_B['initial']} -> {len(SEL_B)} | accuracy {TB.set_index('model').loc[BEST,'test_accuracy']:.4f}
Inflation A - B : {delta.mean():.4f} points d'accuracy en moyenne
Versions {len(df)} | packages {df.package.nunique()} | groupes {df.group_id.nunique()}
Baseline AU-PR (prevalence du test) : {float(np.mean(y_te)):.4f}
Budget d'alertes respecte : {R['protocol_B']['budget_feasible']}
AUC ecosysteme seul {auc_eco:.4f} vs modele {auc_m_:.4f} -> part ecosysteme {share:.1%}
Features proxy d'ecosysteme retirees avant selection : {len(ECO_PROXIES)}
Features proxy restant dans la selection : {len(proxy)} sur {len(SEL_B)}
Budget de features declare a priori : {CFG['final_feature_budget']}
Latence inference batch=1 : {LAT['inference']['1']['median_ms']:.2f} ms | acces registre {LAT['registry_fetch_and_feature_extraction']['median_ms']:.0f} ms
Calibration : Brier {CAL['brier']:.4f} | ECE {CAL['ece_10bins']:.4f}
Robustesse bruit 10% : AUC {ROB['label_noise_10pct']['auc_roc_median']:.4f} (IQR {ROB['label_noise_10pct']['auc_roc_iqr']:.4f})
Robustesse corruption 20% : AUC {ROB['feature_corruption_20pct']['auc_roc_median']:.4f} (IQR {ROB['feature_corruption_20pct']['auc_roc_iqr']:.4f})
---------------------------------------------------------------------
ABLATIONS — ce qui decide de la contribution
Cout de la reduction (espace complet -> 18) en AU-PR : {ABL['cost_of_reduction_au_pr']:+.4f}
Ecart avec le meilleur selecteur alternatif a k egal : {ABL['matched_budget']['gain_over_best_alternative']:+.4f}
   superieur au bruit ? {ABL['matched_budget']['gain_exceeds_ci']}
Ordre, mesure AVANT budget : ecart {ABL['order_before_budget']['spread_au_pr']:.4f}, distinguable {ABL['order_before_budget']['distinguishable']}
Ordre, mesure APRES budget : ecart {ABL['order_after_budget']['spread_au_pr']:.4f}
Stabilite de la selection (Jaccard moyen) : {ABL['selection_stability']['mean_jaccard']:.3f}
Features retenues dans >=80% des tirages : {ABL['selection_stability']['n_features_selected_80pct']}
Variance sur l'espace complet : jaccard avec/sans = {ABL['variance_on_full_space']['jaccard']:.3f}
=====================================================================""")
if y:
    print("RENDEMENT DE COLLECTE — a reporter tel quel en Section 2.2 :")
    for k, s in y.items():
        print(f"  {k:15s} cible {s['target']:5d} -> obtenu {s['obtained']:5d} "
              f"({s['attempted']:6d} tentatives, {s['shortfall']:5d} manquants)")
    print("  Les manquants viennent des packages retires du registre (404), non d'un choix.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 1.9 MB/s eta 0:00:00
Mounted at /content/drive
Sorties : /content/drive/MyDrive/JCP_revision
Jeu charge depuis le cache : /content/drive/MyDrive/JCP_revision/dataset_v5.csv

Dimensions : (4010, 470)
ecosystem  label
npm        0         597
           1        2326
pypi       0         998
           1          89
dtype: int64

Espace de features : 464 (vman 414, pit 42, cur 8)

Appariement temporel : 4010 -> 3330 versions | {'npm': ['2024-03-22', '2026-08-05'], 'pypi': ['2022-01-10', '2026-07-19']}


familles de noms:   0%|          | 0/826 [00:00<?, ?it/s]

3330 versions | 1814 packages | 1490 groupes | plus grand : 137 (4.1%)
{'n_versions': 3330, 'n_packages': 1814, 'n_groups': 1490, 'largest_group': 137, 'similar_name_pairs': 11, 'windows': {'npm': ['2024-03-22', '2026-08-05'], 'pypi': ['2022-01-10', '2026-07-19']}, 'composition': {'npm_0': 413, 'npm_1': 2092, 'pypi_0': 746, 'pypi_1': 79}}

PROTOCOLE A
        initial -> 464
       variance -> 145
    mutual_info -> 100
         boruta -> 85
            rfe -> 30
     corr_prune -> 25
         budget -> 18
       model  train_accuracy  test_accuracy  test_f1_pos  test_mcc  test_auc_roc  test_au_pr  au_pr_baseline  test_fpr  TN  FP  FN  TP
    CatBoost        1.000000       0.980792     0.985185  0.958003      0.993357    0.997022        0.651861  0.017241 285   5  11 532
RandomForest        0.995194       0.975990     0.981308  0.948601      0.991154    0.996277        0.651861  0.006897 288   2  18 525
    Stacking        1.000000       0.980792     0.985158  0.958163      0.991319    

In [ ]:
# =============================================================================
# PATCH [E'] — a executer apres la cellule v7, meme runtime.
#
# Faiblesse du baseline aleatoire de la v7 : rng.choice(Xtr.columns, 18) pioche
# dans les 40 features restant apres le pre-filtre ecosysteme. Tirer 18 sur 40,
# c'est prendre 45% de l'espace : le plancher est artificiellement haut, et la
# comparaison sous-estime l'apport de toute selection.
#
# Ici le tirage se fait sur l'espace point-in-time COMPLET (456), qui est
# l'espace dont un praticien disposerait sans methode de selection.
# =============================================================================
ECO_SAVE_E = list(ECO_PROXIES)
ECO_PROXIES = []
prep_e, F_e = make_prep(train, ("vman_", "pit_"))
Xtr_e, Xte_e = prep_e(train), prep_e(test)
ECO_PROXIES = ECO_SAVE_E
print(f"Espace complet point-in-time : {Xtr_e.shape[1]} features")

def ev_e(feats):
    feats = [f for f in feats if f in Xtr_e.columns]
    m = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, l2_leaf_reg=3,
                           verbose=0, random_state=SEED, auto_class_weights="Balanced")
    m.fit(Xtr_e[feats], y_tr); s = m.predict_proba(Xte_e[feats])[:, 1]
    return {"auc_roc": roc_auc_score(y_te, s),
            "au_pr": average_precision_score(y_te, s)}

K = CFG["final_feature_budget"]
rng_e = np.random.default_rng(SEED)
vals, n_proxy_drawn = [], []
for _ in range(CFG["n_random_baseline"]):
    fs = list(rng_e.choice(Xtr_e.columns, size=K, replace=False))
    e = ev_e(fs); vals.append((e["auc_roc"], e["au_pr"]))
    n_proxy_drawn.append(sum(1 for f in fs if f in ECO_SAVE_E))
V = np.array(vals)

RAND_FULL = {
    "space_size": int(Xtr_e.shape[1]), "k": K,
    "n_draws": CFG["n_random_baseline"],
    "auc_roc_median": float(np.median(V[:, 0])),
    "auc_roc_p05": float(np.percentile(V[:, 0], 5)),
    "auc_roc_p95": float(np.percentile(V[:, 0], 95)),
    "au_pr_median": float(np.median(V[:, 1])),
    "au_pr_p05": float(np.percentile(V[:, 1], 5)),
    "au_pr_p95": float(np.percentile(V[:, 1], 95)),
    "mean_ecosystem_proxies_per_draw": float(np.mean(n_proxy_drawn)),
    "note": ("tirage sur l'espace point-in-time complet ; chaque tirage contient "
             "en moyenne le nombre de proxies d'ecosysteme indique, ce qui gonfle "
             "sa performance apparente"),
}
ours_aupr = ABL["matched_budget"]["table"][0]["au_pr"] \
    if ABL["matched_budget"]["table"][0]["selector"].startswith("Hybrid") else None
if ours_aupr is None:
    ours_aupr = [r for r in ABL["matched_budget"]["table"]
                 if r["selector"].startswith("Hybrid")][0]["au_pr"]
gain_full = ours_aupr - RAND_FULL["au_pr_median"]
CI_W = ABL["au_pr_ci_width"]

print(json.dumps(RAND_FULL, indent=2))
print(f"\nHybrid {ours_aupr:.4f}  vs  Random-{K} sur {Xtr_e.shape[1]} features "
      f"{RAND_FULL['au_pr_median']:.4f}")
print(f"ecart = {gain_full:+.4f} | IC95 = {CI_W:.4f} -> "
      + ("l'apport de la selection est superieur au bruit"
         if gain_full >= CI_W else
         "l'apport de la selection N'EST PAS distinguable du bruit : l'espace de "
         "metadonnees est massivement redondant, et c'est le resultat a publier"))

RAND_FULL["gain_over_random_full_space"] = float(gain_full)
RAND_FULL["gain_exceeds_ci"] = bool(gain_full >= CI_W)
ABL["random_baseline_full_space"] = RAND_FULL
R["ablations"] = ABL
json.dump(clean(R), open(f"{OUT}/revision_results.json", "w"), indent=2, default=str)

# ---------------------------------------------------------------------------
# Complement : courbe de budget sur l'espace COMPLET, jusqu'a k = 3
# pour situer le point de saturation reel (la v7 s'arretait a k = 5)
# ---------------------------------------------------------------------------
print("\nCourbe de budget etendue (espace complet, selection par importance CatBoost)")
cb_e = CatBoostClassifier(iterations=300, learning_rate=0.1, depth=6, verbose=0,
                          random_state=SEED, auto_class_weights="Balanced").fit(Xtr_e, y_tr)
rank_e = pd.Series(cb_e.get_feature_importance(), index=Xtr_e.columns
                   ).sort_values(ascending=False)
rows = []
for k in [1, 2, 3, 5, 8, 10, 15, 18, 25, 40, 80, Xtr_e.shape[1]]:
    if k > Xtr_e.shape[1]: continue
    rows.append({"k": k, **ev_e(list(rank_e.head(k).index))})
SW2 = pd.DataFrame(rows)
print(SW2.to_string(index=False))
sat = SW2[SW2.au_pr >= SW2.au_pr.max() - CI_W].k.min()
print(f"-> saturation atteinte des k = {sat} features "
      f"(premier k dont l'AU-PR est a moins d'un IC du maximum)")
SW2.to_csv(f"{OUT}/figure6b_budget_sweep_full_space.csv", index=False)
ABL["budget_sweep_full_space"] = {"table": SW2.to_dict("records"),
                                  "saturation_k": int(sat)}
R["ablations"] = ABL
json.dump(clean(R), open(f"{OUT}/revision_results.json", "w"), indent=2, default=str)
print(f"\nRe-exporte dans {OUT}/revision_results.json")

Espace complet point-in-time : 456 features
{
  "space_size": 456,
  "k": 18,
  "n_draws": 20,
  "auc_roc_median": 0.814411176184779,
  "auc_roc_p05": 0.7016541975998789,
  "auc_roc_p95": 0.938868876268893,
  "au_pr_median": 0.9053551503473616,
  "au_pr_p05": 0.8246269902980297,
  "au_pr_p95": 0.9731954209117277,
  "mean_ecosystem_proxies_per_draw": 16.5,
  "note": "tirage sur l'espace point-in-time complet ; chaque tirage contient en moyenne le nombre de proxies d'ecosysteme indique, ce qui gonfle sa performance apparente"
}

Hybrid 0.9912  vs  Random-18 sur 456 features 0.9054
ecart = +0.0858 | IC95 = 0.0055 -> l'apport de la selection est superieur au bruit

Courbe de budget etendue (espace complet, selection par importance CatBoost)
  k  auc_roc    au_pr
  1 0.695539 0.822214
  2 0.936123 0.966247
  3 0.926212 0.961590
  5 0.945268 0.977445
  8 0.956629 0.983863
 10 0.961086 0.986451
 15 0.974949 0.991625
 18 0.968732 0.989202
 25 0.972164 0.990597
 40 0.974395 0.990975
 80 0.98108

In [1]:
from scipy.stats import binomtest
import itertools, numpy as np, pandas as pd

def mcnemar_paired(y, pa, pb):
    """Test exact de McNemar sur les predictions discordantes."""
    a_ok, b_ok = (pa == y), (pb == y)
    b = int(np.sum(a_ok & ~b_ok)); c = int(np.sum(~a_ok & b_ok))
    p = binomtest(min(b, c), b + c, 0.5).pvalue if (b + c) else 1.0
    return {"b": b, "c": c, "p_exact": float(p)}

def paired_bootstrap_aupr(y, sa, sb, n=2000, seed=SEED):
    """IC95 sur la DIFFERENCE d'AU-PR, reechantillonnage apparie."""
    rng = np.random.default_rng(seed); d = []
    y, sa, sb = np.asarray(y), np.asarray(sa), np.asarray(sb)
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) < 2: continue
        d.append(average_precision_score(y[i], sa[i]) -
                 average_precision_score(y[i], sb[i]))
    d = np.array(d)
    return {"mean_diff": float(d.mean()),
            "ci95": [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))],
            "p_two_sided": float(2 * min((d <= 0).mean(), (d >= 0).mean()))}

# --- scores de chaque selecteur, refit sur le meme train -------------------
def scores_for(feats):
    m = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, l2_leaf_reg=3,
                           verbose=0, random_state=SEED, auto_class_weights="Balanced")
    m.fit(Xtr[feats], y_tr)
    return m.predict_proba(Xte[feats])[:, 1]

SEL_SETS = {nm: fs for nm, fs in cands.items()}      # cands vient de la section E
SCORES = {nm: scores_for(fs) for nm, fs in SEL_SETS.items()}
PRED = {nm: (s >= 0.5).astype(int) for nm, s in SCORES.items()}

REF = "Hybrid pipeline (ours)"
rows = []
for nm in SEL_SETS:
    if nm == REF: continue
    mc = mcnemar_paired(y_te, PRED[REF], PRED[nm])
    pb = paired_bootstrap_aupr(y_te, SCORES[REF], SCORES[nm])
    rows.append({"comparison": f"{REF} vs {nm}", **mc,
                 "delta_au_pr": pb["mean_diff"],
                 "ci95_low": pb["ci95"][0], "ci95_high": pb["ci95"][1],
                 "p_bootstrap": pb["p_two_sided"]})

# --- cinq features contre dix-huit ----------------------------------------
saved = CFG["final_feature_budget"]
CFG["final_feature_budget"] = 5
f5, _ = select(Xtr, y_tr, stages=tuple(STAGES_SEQ), verbose=False)
CFG["final_feature_budget"] = saved
s5 = scores_for(f5)
pb = paired_bootstrap_aupr(y_te, SCORES[REF], s5)
rows.append({"comparison": "18 features vs 5 features",
             **mcnemar_paired(y_te, PRED[REF], (s5 >= 0.5).astype(int)),
             "delta_au_pr": pb["mean_diff"],
             "ci95_low": pb["ci95"][0], "ci95_high": pb["ci95"][1],
             "p_bootstrap": pb["p_two_sided"]})

PAIRED = pd.DataFrame(rows)
n_comp = len(PAIRED)
PAIRED["p_exact_bonf"] = (PAIRED["p_exact"] * n_comp).clip(upper=1.0)
PAIRED["p_bootstrap_bonf"] = (PAIRED["p_bootstrap"] * n_comp).clip(upper=1.0)
print(PAIRED.to_string(index=False))
PAIRED.to_csv(f"{OUT}/table7b_paired_tests.csv", index=False)
R["paired_tests"] = PAIRED.to_dict("records")
json.dump(clean(R), open(f"{OUT}/revision_results.json", "w"), indent=2, default=str)

NameError: name 'SEED' is not defined

In [2]:
dfn = df[df.ecosystem == "npm"].reset_index(drop=True)
print(f"NPM seul : {len(dfn)} versions | {dfn.label.mean():.1%} malveillant "
      f"| {dfn.group_id.nunique()} groupes")

i_tv, i_te_n = next(GroupShuffleSplit(1, test_size=CFG["test_frac"], random_state=SEED)
                    .split(dfn, dfn.label, groups=dfn.group_id))
tv_n, te_n = dfn.iloc[i_tv].copy(), dfn.iloc[i_te_n].copy()
i_tr_n, i_va_n = next(GroupShuffleSplit(1, test_size=CFG["val_frac"], random_state=SEED)
                      .split(tv_n, tv_n.label, groups=tv_n.group_id))
tr_n, va_n = tv_n.iloc[i_tr_n].copy(), tv_n.iloc[i_va_n].copy()
assert not (set(tr_n.group_id) & set(te_n.group_id))

# aucun proxy d'ecosysteme a retirer : un seul ecosysteme
ECO_SAVE_N = list(ECO_PROXIES); ECO_PROXIES = []
prep_n, _ = make_prep(tr_n, ("vman_", "pit_"))
Xtr_n, Xva_n, Xte_n = prep_n(tr_n), prep_n(va_n), prep_n(te_n)
ytr_n, yva_n, yte_n = tr_n.label.values, va_n.label.values, te_n.label.values
SEL_N, LOG_N = select(Xtr_n, ytr_n)
print("NPM-only :", LOG_N, "\n", SEL_N)

rows_n = []
for nm, m in build_models().items():
    m.fit(Xtr_n[SEL_N], ytr_n)
    s = m.predict_proba(Xte_n[SEL_N])[:, 1]
    rows_n.append({"protocol": "B_npm_only", "model": nm, **metrics(yte_n, s)})
TN_ = pd.DataFrame(rows_n)
print(TN_.to_string(index=False))
ECO_PROXIES = ECO_SAVE_N
R["protocol_B_npm_only"] = {"selection_log": LOG_N, "selected": SEL_N,
                            "table": TN_.to_dict("records"),
                            "n_versions": int(len(dfn)),
                            "malicious_ratio": float(dfn.label.mean())}

NameError: name 'df' is not defined